# Channel Analytics — Dashboard reports in Excel

Writes the analysis of the Retail and HS dashboards as six Excel workbooks:

| File | Scope | Values |
|---|---|---|
| `Retail_Analysis_Gross_<Mon>_<year>.xlsx` | Retail | Gross sales |
| `Retail_Analysis_Net_<Mon>_<year>.xlsx` | Retail | Net sales (your rebates) |
| `HS_with_FD_Analysis_Gross_…` / `…_Net_…` | HS (Retail + Project), FD included | Gross / Net |
| `HS_without_FD_Analysis_Gross_…` / `…_Net_…` | HS (Retail + Project), FD left out | Gross / Net |

**How to run:** set the rebates and the data source in cell 1, then *Run All*.
The data are the dashboard input files written by `Channel_Analytics_v9.ipynb`
(Retail_Sales_OB_Backlog_Combined.xlsx and H&D_Overall_Sales_OB_Backlog_Combined_ppcfd removed.xlsx),
the Material Master, the Data Dictionary and the target files — the same files the dashboards read.
Each workbook starts with an **About** sheet (period, basis, rebates, rules) and an **Overview** sheet.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 0 · SETUP — installs anything missing (run once per machine)
# ════════════════════════════════════════════════════════════════════
import importlib
import subprocess
import sys

for pip_name, import_name in {"pandas": "pandas", "numpy": "numpy", "openpyxl": "openpyxl", "xlsxwriter": "xlsxwriter",
                              "pyarrow": "pyarrow", "msal": "msal", "requests": "requests", "python-dotenv": "dotenv",
                              "python-calamine": "python_calamine"}.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"  ⏳ installing {pip_name} ...", end=" ", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "--quiet"])
        print("done")
print("✅ Packages ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 1 · SETTINGS — the only cell you normally edit
# ════════════════════════════════════════════════════════════════════
import io
import os
import re
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

# --- Rebates for the Net files: Net = Gross × (1 − rebate), by Retail / Project and year ------------
REBATES = {
    "Retail": {2026: 0.135, 2025: 0.122, 2024: 0.145, 2023: 0.14, 2022: 0.13},
    "Project": {2026: 0.051, 2025: 0.045, 2024: 0.045, 2023: 0.045, 2022: 0.045},
}

# --- Period -------------------------------------------------------------------------------------------
AS_OF = None                 # None = the last invoice date in the data; or a date in quotes, e.g. "2026-10-31"

# --- Where the input files are -----------------------------------------------------------------------
SOURCE = "sharepoint"        # "sharepoint" (needs the .env with SP_TENANT_ID / SP_CLIENT_ID / SP_CLIENT_SECRET)
                             # or "local" (files in LOCAL_DIR, e.g. your OneDrive-synced folder)
ENV_FILE_PATH = ""           # .env location if it is not next to this notebook
LOCAL_DIR = r""              # folder that holds the files below when SOURCE = "local"

SITE_MAIN = "schneiderelectric.sharepoint.com:/sites/HDIndiaTeamGroup"
_ROOT = "Test - haripriya"
_MASTER = f"{_ROOT}/Data Downloads - Input Files/Master data"
_DASH = f"{_ROOT}/Dashboard Input Files/H&D Channel"
INPUTS = {   # key: (SharePoint path, local file name)
    "retail": (f"{_DASH}/Retail_Sales_OB_Backlog_Combined.xlsx", "Retail_Sales_OB_Backlog_Combined.xlsx"),
    "hs": (f"{_DASH}/H&D_Overall_Sales_OB_Backlog_Combined_ppcfd removed.xlsx",
           "H&D_Overall_Sales_OB_Backlog_Combined_ppcfd removed.xlsx"),
    "material": (f"{_MASTER}/Material Master Data.xlsx", "Material Master Data.xlsx"),
    "dictionary": (f"{_MASTER}/Data Dictionary.xlsx", "Data Dictionary.xlsx"),
    "targets": (f"{_MASTER}/Master Targets File.xlsx", "Master Targets File.xlsx"),
    "qtod": (f"{_MASTER}/Targets File/QTOD and ATOD file.xlsx", "QTOD and ATOD file.xlsx"),
    "hs_targets": (f"{_MASTER}/Master Targets File (Retail+Project).xlsx", "Master Targets File (Retail+Project).xlsx"),
}

# --- Output -------------------------------------------------------------------------------------------
OUTPUT_DIR = "./Dashboard Reports"
UPLOAD_TO_SHAREPOINT = False
SP_OUTPUT_FOLDER = f"{_ROOT}/Final Reports/Dashboard Reports"
CACHE_DIR = "./local_cache_reports"     # parsed input files are cached here (re-read only when a file changes)

# --- Rules (as the dashboards) --------------------------------------------------------------------------
FD_HELIOS = ["HDCFD", "PPCFD", "HDFDA", "HDFDB", "HDFDS"]
MANUAL_OLD_PARTNER_IDS = {"500073231"}   # always "Old" in the policy tracker
FLAT_NEW_PARTNER_TARGET = 2_500_000      # policy tracker: assigned target of a new partner (gross)
BACKLOG_OLD_DAYS = 50                     # backlog older than this is flagged
UNIT, UNIT_LABEL = 1e6, "Mn"              # values in the reports: INR Million
FOCUS_GROUPS = ["Wiser", "Miluz Zeta", "Unica Pure", "Livia", "AvatarOn", "Easy9 6kA", "Easy9 10kA", "Easy9 RCCB", "Easy9 DB"]

for _d in (OUTPUT_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)
print(f"✅ Settings | source: {SOURCE} | Net rebates {REBATES['Retail']} (Retail), {REBATES['Project']} (Project)")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 2 · SHAREPOINT (only when SOURCE = "sharepoint") — same client as Channel_Analytics_v9
# ════════════════════════════════════════════════════════════════════
from urllib.parse import quote

import msal
import requests

GRAPH = "https://graph.microsoft.com/v1.0"
SIMPLE_UPLOAD_LIMIT = 4 * 1024 * 1024    # Graph's limit for a single PUT
UPLOAD_CHUNK = 5 * 1024 * 1024            # multiple of 320 KiB, as Graph requires


class SharePoint:
    def __init__(self):
        self._app = msal.ConfidentialClientApplication(
            os.environ["SP_CLIENT_ID"],
            authority=f"https://login.microsoftonline.com/{os.environ['SP_TENANT_ID']}",
            client_credential=os.environ["SP_CLIENT_SECRET"],
        )
        self._drives = {}

    # --- plumbing ---------------------------------------------------------
    def headers(self):
        token = self._app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        if "access_token" not in token:
            raise PermissionError(f"SharePoint authentication failed: {token.get('error_description', token)}")
        return {"Authorization": f"Bearer {token['access_token']}"}

    def _request(self, method, url, auth=True, attempts=4, **kwargs):
        extra_headers = kwargs.pop("headers", {})
        for attempt in range(1, attempts + 1):
            try:
                hdrs = {**(self.headers() if auth else {}), **extra_headers}
                res = requests.request(method, url, headers=hdrs, timeout=300, **kwargs)
                if res.status_code in (429, 500, 502, 503, 504) and attempt < attempts:
                    raise requests.ConnectionError(f"HTTP {res.status_code}")
                return res
            except (requests.ConnectionError, requests.Timeout) as e:
                if attempt == attempts:
                    raise
                wait = 3 * 2 ** (attempt - 1)
                print(f"   ⚠️ {method} failed ({e}); retrying in {wait}s ...")
                time.sleep(wait)

    def drive_id(self, site=SITE_MAIN):
        if site not in self._drives:
            site_id = self._request("GET", f"{GRAPH}/sites/{site}").json()["id"]
            drives = self._request("GET", f"{GRAPH}/sites/{site_id}/drives").json()["value"]
            self._drives[site] = next(d["id"] for d in drives if d["name"] == "Documents")
        return self._drives[site]

    def _item_url(self, path, site):
        return f"{GRAPH}/drives/{self.drive_id(site)}/root:/{quote(path)}"

    # --- reading ----------------------------------------------------------
    def list_folder(self, folder, site=SITE_MAIN):
        """Items in a folder (each has name, lastModifiedDateTime, @microsoft.graph.downloadUrl)."""
        res = self._request("GET", f"{self._item_url(folder, site)}:/children").json()
        if "value" not in res:
            raise FileNotFoundError(f"SharePoint folder not found: {folder} ({res.get('error', {}).get('message', res)})")
        return res["value"]

    def exists(self, path, site=SITE_MAIN):
        return self._request("GET", self._item_url(path, site)).status_code == 200

    def download_item(self, item):
        return self._request("GET", item["@microsoft.graph.downloadUrl"], auth=False).content

    def download(self, path, site=SITE_MAIN):
        meta = self._request("GET", self._item_url(path, site)).json()
        if "@microsoft.graph.downloadUrl" not in meta:
            raise FileNotFoundError(f"SharePoint file not found: {path}")
        return io.BytesIO(self.download_item(meta))

    def read_excel(self, path, site=SITE_MAIN, **kwargs):
        return pd.read_excel(self.download(path, site), **kwargs)

    # --- writing ----------------------------------------------------------
    def upload(self, path, data, site=SITE_MAIN):
        """Create/replace a file. Uses an upload session for files over 4 MB."""
        if isinstance(data, io.BytesIO):
            data = data.getvalue()
        if len(data) <= SIMPLE_UPLOAD_LIMIT:
            res = self._request("PUT", f"{self._item_url(path, site)}:/content", data=data)
        else:
            session = self._request(
                "POST", f"{self._item_url(path, site)}:/createUploadSession",
                json={"item": {"@microsoft.graph.conflictBehavior": "replace"}},
            ).json()
            if "uploadUrl" not in session:
                raise IOError(f"Could not open upload session for {path}: {session}")
            res = None
            for start in range(0, len(data), UPLOAD_CHUNK):
                chunk = data[start:start + UPLOAD_CHUNK]
                res = self._request("PUT", session["uploadUrl"], auth=False, data=chunk, headers={
                    "Content-Length": str(len(chunk)),
                    "Content-Range": f"bytes {start}-{start + len(chunk) - 1}/{len(data)}",
                })
        if res is None or res.status_code not in (200, 201):
            hint = " — the file is open/locked in SharePoint, close it and re-run" if res is not None and res.status_code == 423 else ""
            raise IOError(f"Upload failed for {path}: HTTP {getattr(res, 'status_code', '?')}{hint}")
        print(f"   ⬆️  {path.split('/')[-1]} → SharePoint ({len(data) / 1e6:.1f} MB)")
        return res

    def delete(self, path, site=SITE_MAIN):
        """Delete a file (to the SharePoint recycle bin). Returns False if it was not there."""
        res = self._request("DELETE", self._item_url(path, site))
        if res.status_code == 404:
            return False
        if res.status_code not in (200, 204):
            raise IOError(f"Delete failed for {path}: HTTP {res.status_code}")
        print(f"   🗑️  {path.split('/')[-1]} deleted (SharePoint recycle bin)")
        return True

    def ensure_folder(self, folder, site=SITE_MAIN):
        parent, _, name = folder.rpartition("/")
        url = f"{self._item_url(parent, site)}:/children" if parent else f"{GRAPH}/drives/{self.drive_id(site)}/root/children"
        self._request("POST", url, json={"name": name, "folder": {}, "@microsoft.graph.conflictBehavior": "fail"})

def _load_env():
    from dotenv import find_dotenv, load_dotenv
    path = ENV_FILE_PATH or find_dotenv(usecwd=True)
    if path:
        load_dotenv(path, override=False)
    missing = [k for k in ("SP_TENANT_ID", "SP_CLIENT_ID", "SP_CLIENT_SECRET") if not os.environ.get(k)]
    if missing:
        raise EnvironmentError(f"SharePoint keys missing: {', '.join(missing)} — put them in the .env file "
                               "(set ENV_FILE_PATH) or use SOURCE = 'local'")


sp = None
if SOURCE == "sharepoint" or UPLOAD_TO_SHAREPOINT:
    _load_env()
    sp = SharePoint()
    print("✅ SharePoint client ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 3 · REPORT STYLE — same look as the Channel Analytics reports
# ════════════════════════════════════════════════════════════════════
from contextlib import contextmanager

THEME = {
    "font": "Calibri",
    "font_size": 10,
    "header_bg": "#1B7A3E",      # column headers (white text)
    "band_bg": "#215967",        # group/section bands (white text)
    "accent": "#3DCD58",         # Schneider green — tab colour, KPI highlights
    "title": "#1B7A3E",
    "subtitle": "#595959",
    "zebra": "#F2F9F4",
    "total_bg": "#DAEFDF",
    "kpi_bg": "#EAF6EF",
    "border": "#BFBFBF",
    "good": ("#C6EFCE", "#276221"),
    "warn": ("#FFEB9C", "#9C5700"),
    "bad": ("#FFC7CE", "#9C0006"),
    # distinct, consistent colours for multi-group tables and tabs
    "series": ["#215967", "#1B7A3E", "#2E75B6", "#7030A0", "#C55A11", "#BF8F00", "#546E7A"],
    "tab_summary": "#3DCD58",
    "tab_detail": "#215967",
    "tab_tracker": "#C55A11",
    "tab_data": "#A6A6A6",
}
NUM_FMT = {
    "inr": "#,##0",
    "mn": "#,##0.0",
    "int": "#,##0",
    "pct": "0.0%",
    "date": "dd-mmm-yyyy",
    "id": "0",
    "plain": "0",
}


def base_formats(workbook):
    """Every named format used by the reports. Keys from the older per-report
    format dicts are kept as aliases so all report code shares this one set."""
    T, N = THEME, NUM_FMT
    base = {"font_name": T["font"], "font_size": T["font_size"], "valign": "vcenter"}
    cell = {**base, "border": 1, "border_color": T["border"]}

    def fmt(**kw):
        return workbook.add_format({**cell, **kw})

    hdr = {"bold": True, "bg_color": T["header_bg"], "font_color": "#FFFFFF", "align": "center", "text_wrap": True}
    band = {"bold": True, "bg_color": T["band_bg"], "font_color": "#FFFFFF", "align": "center"}
    tot = {"bold": True, "bg_color": T["total_bg"], "top": 2}
    zebra = {"bg_color": T["zebra"]}
    f = {
        "title": workbook.add_format({**base, "bold": True, "font_size": 14, "font_color": T["title"]}),
        "subtitle": workbook.add_format({**base, "bold": True, "font_size": 11, "font_color": T["band_bg"]}),
        "note": workbook.add_format({**base, "italic": True, "font_size": 9, "font_color": T["subtitle"]}),
        "hdr": fmt(**hdr),
        "band": fmt(**band),
        "text": fmt(align="left"),
        "text_alt": fmt(align="left", **zebra),
        "inr": fmt(num_format=N["inr"], align="right"),
        "inr_alt": fmt(num_format=N["inr"], align="right", **zebra),
        "mn": fmt(num_format=N["mn"], align="right"),
        "int": fmt(num_format=N["int"], align="center"),
        "pct": fmt(num_format=N["pct"], align="center"),
        "pct_alt": fmt(num_format=N["pct"], align="center", **zebra),
        "id": fmt(num_format=N["id"], align="left"),
        "id_alt": fmt(num_format=N["id"], align="left", **zebra),
        "date": fmt(num_format=N["date"], align="center"),
        "date_alt": fmt(num_format=N["date"], align="center", **zebra),
        "tot_text": fmt(align="left", **tot),
        "tot_inr": fmt(num_format=N["inr"], align="right", **tot),
        "tot_mn": fmt(num_format=N["mn"], align="right", **tot),
        "tot_int": fmt(num_format=N["int"], align="center", **tot),
        "tot_pct": fmt(num_format=N["pct"], align="center", **tot),
        "kpi_label": fmt(bold=True, bg_color=T["kpi_bg"], align="left", font_size=11),
        "kpi_value": fmt(bold=True, align="center", font_size=11, num_format=N["mn"]),
        "kpi_value_int": fmt(bold=True, align="center", font_size=11, num_format=N["int"]),
        "kpi_pct_hero": fmt(bold=True, bg_color=T["accent"], align="center", font_size=13, num_format=N["pct"]),
        "stat_label": fmt(**hdr),
        "stat_value": fmt(bold=True, bg_color=T["kpi_bg"], align="center", font_size=16),
        "runrate": fmt(num_format=N["inr"], align="right", bold=True, bg_color=T["warn"][0], font_color=T["bad"][1]),
        "cf_green": fmt(bg_color=T["good"][0], font_color=T["good"][1], num_format=N["pct"], align="center"),
        "cf_yellow": fmt(bg_color=T["warn"][0], font_color=T["warn"][1], num_format=N["pct"], align="center"),
        "cf_red": fmt(bg_color=T["bad"][0], font_color=T["bad"][1], num_format=N["pct"], align="center"),
        "zebra_fill": workbook.add_format({"bg_color": T["zebra"]}),
    }
    # Aliases used by the Partner Analysis / SSP code
    f.update({
        "hdr_wrap": f["hdr"], "hdr_alt": f["band"], "hdr_pol": f["hdr"], "hdr_green": f["hdr"], "hdr_dark": f["band"],
        "txt_det": f["text"], "dec": f["mn"], "num_mil": f["mn"],
        "num_det": f["inr"], "num_abs": f["inr"], "num_abs_alt": f["inr_alt"], "policy_num": f["inr"],
        "num_id": f["id"], "num_id_alt": f["id_alt"],
        "tot_dec": f["tot_mn"], "tot_det": f["tot_inr"], "tot_num": f["tot_inr"],
        "policy_runrate": f["runrate"],
    })
    return f


@contextmanager
def report_writer(target):
    """pandas ExcelWriter (xlsxwriter) with the house font and date format.
    `target` is a file path or a BytesIO."""
    with pd.ExcelWriter(
        target, engine="xlsxwriter",
        date_format=NUM_FMT["date"], datetime_format=NUM_FMT["date"],
        engine_kwargs={"options": {"nan_inf_to_errors": True}},
    ) as writer:
        default = writer.book.formats[0]          # workbook default cell format
        default.set_font_name(THEME["font"])
        default.set_font_size(THEME["font_size"])
        writer.fmt = base_formats(writer.book)     # handy: writer.fmt["hdr"] ...
        yield writer


# --- column typing --------------------------------------------------------
_PCT_WORDS = ("%", "pct", "percent", "achievement", "growth", "gwt", "grw", "contr", "rebate", "pace")
_MN_WORDS = ("(mn)", " mn", "mnr", "_mn")
_ID_WORDS = ("id", "code", "party", "order", "header", "item", "gstn", "gstin", "invoice", "po", "pin", "material")
_PLAIN_WORDS = ("year", "month", "_y", "_m", "rank", "remaining months")


def column_kind(name, series):
    """Classify a column → 'date' | 'pct' | 'mn' | 'id' | 'plain' | 'inr' | 'text'."""
    n = str(name).strip().lower()
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    if series.dtype == object and len(series.dropna()) and isinstance(series.dropna().iloc[0], (datetime, pd.Timestamp)):
        return "date"
    if not pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        return "text"
    words = re.split(r"[\s_/().-]+", n)
    if any(w in n for w in _PCT_WORDS) and "run rate" not in n:
        return "pct"
    if any(n.endswith(w) or w in n for w in _MN_WORDS):
        return "mn"
    is_amount = any(w in n for w in ("value", "sales", "amt", "amount", "qty", "quantity", "target", "total"))
    if not is_amount and (any(w in words for w in _ID_WORDS) or n.endswith("id")):
        return "id"
    if n in _PLAIN_WORDS or any(n == w or n.endswith(w) for w in ("year", "month")):
        return "plain"
    return "inr"


def _col_width(name, series, kind):
    sample = series.dropna().head(500)
    if kind in ("inr", "mn", "int"):
        longest = len(f"{sample.abs().max():,.0f}") + 2 if len(sample) else 6
    elif kind in ("pct", "date", "plain"):
        longest = 11
    else:
        longest = int(sample.astype(str).str.len().quantile(0.95)) if len(sample) else 6
    return max(8, min(45, max(longest, len(str(name)) * 0.9) + 2))


def write_table(writer, df, sheet_name, title=None, subtitle=None, startrow=0, tab=None,
                freeze_cols=0, totals=False, kinds=None, autofilter=True, zebra=True, header_height=30):
    """Write `df` as a house-style table. Returns the row after the table.

    Without a title the header is on row 1 (safe for files other code reads).
    `kinds` overrides the auto-detected column type, e.g. {"Target": "mn"}."""
    f = writer.fmt
    t0 = time.time()
    print(f"      ✎ sheet '{sheet_name}' — {len(df):,} rows × {len(df.columns)} cols")
    df = df.reset_index(drop=True)
    df.columns = [str(c) for c in df.columns]
    hdr_row = startrow + (2 if title else 0) + (1 if subtitle else 0)
    df.to_excel(writer, sheet_name=sheet_name, startrow=hdr_row + 1, index=False, header=False)
    ws = writer.sheets[sheet_name]
    if title:
        ws.write(startrow, 0, title, f["title"])
    if subtitle:
        ws.write(startrow + (1 if title else 0), 0, subtitle, f["note"])

    kinds = {**{c: column_kind(c, df[c]) for c in df.columns}, **(kinds or {})}
    for i, col in enumerate(df.columns):
        ws.write(hdr_row, i, col, f["hdr"])
        k = kinds[col]
        cell_fmt = {"inr": f["inr"], "mn": f["mn"], "int": f["int"], "pct": f["pct"], "id": f["id"],
                    "date": f["date"], "plain": None, "text": f["text"]}[k]
        ws.set_column(i, i, _col_width(col, df[col], k), cell_fmt)
    ws.set_row(hdr_row, header_height)

    first, last = hdr_row + 1, hdr_row + len(df)
    if len(df) and zebra:
        ws.conditional_format(first, 0, last, len(df.columns) - 1,
                              {"type": "formula", "criteria": f"=MOD(ROW()-{first},2)=1", "format": f["zebra_fill"]})
    if autofilter and len(df.columns):
        ws.autofilter(hdr_row, 0, max(last, hdr_row), len(df.columns) - 1)
    ws.freeze_panes(hdr_row + 1, freeze_cols)
    if tab:
        ws.set_tab_color(tab)

    end = last + 1
    if totals and len(df):
        ws.write(end, 0, "Grand Total", f["tot_text"])
        for i, col in enumerate(df.columns[1:], start=1):
            k = kinds[col]
            if k in ("inr", "mn", "int"):
                ws.write(end, i, float(df[col].sum()), f[f"tot_{k}"])
            else:
                ws.write_blank(end, i, None, f["tot_text"])
        end += 1
    if time.time() - t0 > 5:
        print(f"        sheet '{sheet_name}' written in {time.time() - t0:.0f}s")
    return end + 1


def kpi_cards(ws, f, row, cards, per_row=3, span=2, gap=1):
    """Stat cards: list of (label, value). Returns the next free row."""
    for i, (label, value) in enumerate(cards):
        r = row + (i // per_row) * 3
        c = (i % per_row) * (span + gap)
        ws.merge_range(r, c, r, c + span - 1, label, f["stat_label"])
        ws.merge_range(r + 1, c, r + 1, c + span - 1, value, f["stat_value"])
    return row + ((len(cards) - 1) // per_row + 1) * 3


def apply_3tier_cf(ws, row_start, row_end, col_idx, f):
    """≥100% green, 80–99% amber, <80% red — same thresholds on every report."""
    ws.conditional_format(row_start, col_idx, row_end, col_idx, {"type": "cell", "criteria": ">=", "value": 1.0, "format": f["cf_green"]})
    ws.conditional_format(row_start, col_idx, row_end, col_idx, {"type": "cell", "criteria": "between", "minimum": 0.8, "maximum": 0.9999, "format": f["cf_yellow"]})
    ws.conditional_format(row_start, col_idx, row_end, col_idx, {"type": "cell", "criteria": "<", "value": 0.8, "format": f["cf_red"]})


def save_report(build, filename, sp_folder=None, local_dir=None):
    """Run build(writer) on an in-memory workbook, then upload it to
    SharePoint (`sp_folder`) and/or save it in `local_dir`. Returns where it went."""
    print(f"   📄 building {filename}")
    t0 = time.time()
    buf = io.BytesIO()
    with report_writer(buf) as writer:
        build(writer)
    data = buf.getvalue()
    print(f"   📦 {filename} ready ({len(data) / 1e6:.1f} MB, {time.time() - t0:.0f}s) — saving")
    targets = []
    if local_dir:
        os.makedirs(local_dir, exist_ok=True)
        path = os.path.join(local_dir, filename)
        with open(path, "wb") as fh:
            fh.write(data)
        print(f"   💾 {path}")
        targets.append(path)
    if sp_folder:
        sp.upload(f"{sp_folder}/{filename}", data)
        targets.append(f"SharePoint: {sp_folder}/{filename}")
    return "; ".join(targets)


print("✅ Report style ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 4 · TARGET RULES — organize_targets.py (the rules behind Channel_Targets_<year>.xlsx)
# ════════════════════════════════════════════════════════════════════
import types

def _target_split_tools():
    """organize_targets.py, wrapped so its names stay out of the notebook's globals."""
    import datetime as _dt
    import re
    import numpy as np
    import pandas as pd

    SALES_COLUMNS = ["Year", "Month", "GTM", "Branch", "Region", "HeliosCode", "Family", "Partner", "Gross"]

    ACTIVITY = {"HDCWD": "WD", "HSCWD": "WD", "HDCFD": "FD", "PPCFD": "FD", "HDFDA": "FD", "HDFDB": "FD", "HDFDS": "FD",
                "HDCOS": "COS", "HSCOS": "COS", "HDSPE": "SPE", "HSSPE": "SPE", "PREVC": "EVC"}
    ACTIVITIES = ["WD", "FD", "COS", "SPE", "EVC"]
    # product groups as in the 'Product Targets' sheet, with the activity each belongs to
    GROUPS = [("Unica Pure", "WD"), ("Livia", "WD"), ("AvatarOn", "WD"), ("Miluz Zeta", "WD"), ("WD- All others", "WD"),
              ("Easy9 6kA", "FD"), ("Easy9 10kA", "FD"), ("Easy9 RCCB", "FD"), ("Easy9 DB", "FD"), ("FD- All others", "FD"),
              ("Wiser", "COS"), ("COS- All others", "COS"), ("SPE", "SPE"), ("EVC", "EVC")]
    GROUP_ACTIVITY = dict(GROUPS)
    FOCUS = {"Miluz Zeta": "Miluz Zeta", "Wiser": "Wiser"}      # monthly target column -> product group
    FOCUS_FAMILY = {"Zeta Target": "Miluz Zeta", "Wiser Target": "Wiser"}   # QTOD columns

    # branch spellings in the target files -> the branch the sales data uses
    BRANCH_ALIASES = {"NCR(DEL+ROH)": "NCR", "NCR(NOI+GHAZ)": "NOIDA-GHZ", "PNQ": "PUNE", "NAN": "NASHIK", "TN2": "TN-2",
                      "TN -1": "TN-1", "TN-1 ": "TN-1", "AP ": "AP", "ap": "AP", "UP": "UP-1", "MP": "INDORE"}
    SSP_ALIASES = {"Katson Agency": "Kaston Agency", "Balaji Sales": "Balaji Sales Corp"}
    QUARTERS = {"Q1": (1, 2, 3), "Q2": (4, 5, 6), "Q3": (7, 8, 9), "Q4": (10, 11, 12)}
    MIN_PARTNER_SALES = 1e5          # partners below this base-year sales get their branch's product mix


    def product_group(activity, family):
        f = str(family or "")
        if activity == "WD":
            if f in ("Unica Pure", "Unica Pure Plus"):
                return "Unica Pure"
            return f if f in ("Livia", "AvatarOn", "Miluz Zeta") else "WD- All others"
        if activity == "FD":
            if f.startswith("Easy9 MCB 6kA"):
                return "Easy9 6kA"
            if f.startswith("Easy9 MCB 10kA"):
                return "Easy9 10kA"
            if f == "Easy9 RCCB":
                return "Easy9 RCCB"
            if f.startswith("Easy9 DB") or f in ("Easy9 Enclosure", "Easy9 Metal P and S"):
                return "Easy9 DB"
            return "FD- All others"
        if activity == "COS":
            return "Wiser" if f == "Wiser" else "COS- All others"
        return activity if activity in ("SPE", "EVC") else None


    def canon_branch(b):
        if b is None or (isinstance(b, float) and np.isnan(b)) or str(b).strip() in ("", "0", "nan"):
            return None
        s = str(b).strip()
        return BRANCH_ALIASES.get(s, BRANCH_ALIASES.get(s.upper(), s))


    def clean_code(v):
        return re.sub(r"\.0$", "", str(v).strip()) if pd.notna(v) else ""


    def num(s):
        return pd.to_numeric(s, errors="coerce")


    # ====================================================================== sales basis
    def prepare_sales(sales):
        s = sales.copy()
        s["Activity"] = s["HeliosCode"].astype(str).str.strip().str.upper().map(ACTIVITY)
        s["Group"] = [product_group(a, f) for a, f in zip(s["Activity"], s["Family"])]
        retail = s["GTM"] == "Retail"          # the aliases are Retail spellings; Project uses NS Branches names (NAN, MP ...)
        s.loc[retail, "Branch"] = s.loc[retail, "Branch"].map(canon_branch)
        s["Partner"] = s["Partner"].map(clean_code)
        s["Gross"] = num(s["Gross"]).fillna(0.0)
        return s[s["Group"].notna()]


    def shares(df, keys, col, weight="Gross"):
        """Share of `col` within `keys` (e.g. month within branch)."""
        g = df.groupby(keys + [col])[weight].sum()
        tot = g.groupby(level=list(range(len(keys)))).transform("sum")
        return (g / tot.where(tot > 0)).fillna(0.0)


    def lookup_share(table, key, values, fallback):
        """{value: share} for one key, or the fallback table's shares when the key has no sales."""
        try:
            s = table.loc[key]
            if s.sum() > 0:
                return s.reindex(values, fill_value=0.0)
        except KeyError:
            pass
        return fallback.reindex(values, fill_value=0.0)


    def contribution(sales, base_year):
        """Base-year sales contribution tables (values in rupees and shares)."""
        s = sales[sales["Year"] == base_year]
        out = {}

        def table(part, keys, col, name):
            v = part.groupby(keys + [col])["Gross"].sum().unstack(col, fill_value=0.0)
            if col == "Month":
                v.columns = [_dt.date(2000, int(m), 1).strftime("%b") for m in v.columns]
            tot = v.sum(axis=1)
            sh = v.div(tot.where(tot > 0), axis=0).fillna(0.0)
            sh.columns = [f"{c} %" for c in sh.columns]
            out[name] = pd.concat([v.assign(Total=tot), sh], axis=1).reset_index()

        for g in ("Retail", "Project"):
            part = s[s["GTM"] == g]
            if part.empty:
                continue
            table(part, ["Region", "Branch"], "Month", f"{g} branch x month")
            table(part, ["Region", "Branch"], "Activity", f"{g} branch x activity")
            table(part, ["Region", "Branch"], "Group", f"{g} branch x product")
            table(part, ["Region"], "Group", f"{g} region x product")
        v = s.groupby(["Region", "Branch", "GTM"])["Gross"].sum().unstack("GTM", fill_value=0.0)
        tot = v.sum(axis=1)
        out["Retail vs Project"] = v.assign(Total=tot, **{f"{c} %": v[c] / tot.where(tot > 0) for c in v.columns}).reset_index()
        p = s[s["GTM"] == "Retail"].groupby(["Branch", "Partner", "Group"])["Gross"].sum().unstack("Group", fill_value=0.0)
        out["Retail partner x product"] = p.assign(Total=p.sum(axis=1)).sort_values("Total", ascending=False).reset_index()
        return out


    # ====================================================================== organise
    class Issues(list):
        def add(self, sheet, what, action, rows=""):
            self.append({"Source sheet": sheet, "Problem": what, "What was done": action, "Rows / values": str(rows)[:500]})


    def retail_targets(master, year, issues):
        """Retail yearly ('Branch Targets') and monthly ('Monthly Targets' = the monthly commit) branch targets, as given,
        plus the monthly Miluz Zeta / Wiser product targets of the same sheet. Yearly and monthly are separate targets:
        the months are never derived from the yearly target and need not add up to it."""
        bt = master["Branch Targets"].copy()
        bt["Branch"] = bt["Branch"].map(canon_branch)
        bt["Target"], bt["Year"] = num(bt["Target"]), num(bt["Year"])
        region = dict(zip(bt["Branch"], bt["Zone"]))
        yearly = bt[bt["Year"] == year].groupby("Branch")["Target"].sum(min_count=1).dropna()
        if bt.loc[bt["Year"] == year - 1, "Target"].isna().all() and (bt["Year"] == year - 1).any():
            issues.add("Branch Targets", f"{year - 1} rows have no target", "ignored")

        mt = master["Monthly Targets"].iloc[:, :6].copy()
        mt.columns = ["Zone", "Branch", "Month", "Target", "Miluz Zeta", "Wiser"]
        renamed = sorted({b for b in mt["Branch"].dropna().astype(str).str.strip() if canon_branch(b) != b})
        if renamed:
            issues.add("Monthly Targets", "branch names that differ from the sales data",
                       "renamed: " + ", ".join(f"{b} -> {canon_branch(b)}" for b in renamed))
        side = master["Monthly Targets"].iloc[:, 6:].dropna(how="all")
        if not side.empty:
            issues.add("Monthly Targets", "a second table to the right of the data (e.g. 'Oct WD / FD / Wiser Plan', in lakh)",
                       "not read; keep only the data columns in this sheet", f"{len(side)} rows")
        mt["Branch"] = mt["Branch"].map(canon_branch)
        mt["Month"] = pd.to_datetime(mt["Month"], errors="coerce")
        mt = mt[mt["Month"].dt.year == year]
        for c in ("Target", "Miluz Zeta", "Wiser"):
            mt[c] = num(mt[c])
        monthly = mt.groupby(["Branch", "Month"])[["Target", "Miluz Zeta", "Wiser"]].sum(min_count=1).dropna(subset=["Target"])
        have = sorted(monthly.index.get_level_values(1).month.unique())
        if len(have) < 12:
            missing = [_dt.date(year, m, 1).strftime("%b") for m in range(1, 13) if m not in have]
            issues.add("Monthly Targets", f"no monthly targets yet for {', '.join(missing)}",
                       "left empty (monthly targets are never derived from the yearly target) — add them when set")
        for c in ("Miluz Zeta", "Wiser"):
            blank = mt[mt["Target"].notna() & mt[c].isna()]
            if len(blank):
                issues.add("Monthly Targets", f"blank {c} target", "no product target for that month",
                           ", ".join(f"{b} {d:%b}" for b, d in zip(blank["Branch"], blank["Month"])))
        for b in sorted(set(monthly.index.get_level_values(0)) - set(region)):
            issues.add("Monthly Targets", f"branch {b} is not in 'Branch Targets'", "region left blank")
        tm = monthly["Target"].groupby(level=0).sum()
        over = [(b, tm[b], yearly[b]) for b in tm.index if b in yearly and tm[b] > yearly[b]]
        if over:
            issues.add("Monthly Targets", "monthly targets already add up to more than the yearly target",
                       "both kept as given (they are separate targets) — just for information",
                       "; ".join(f"{b}: months {t / 1e7:,.2f} Cr vs yearly {y / 1e7:,.2f} Cr" for b, t, y in over))

        y_rows = [(year, region.get(b), b, "Retail", "All", v, "Net", "Given (Branch Targets)") for b, v in yearly.items()]
        m_rows, p_rows = [], []
        for (b, m), r in monthly.iterrows():
            m_rows.append((year, m.date(), region.get(b), b, "Retail", "All", r["Target"], "Net", "Given (Monthly Targets)"))
            for col, grp in FOCUS.items():
                if pd.notna(r[col]) and r[col] != 0:
                    p_rows.append((year, m.date(), region.get(b), b, "Retail", grp, float(r[col]), "Net", "Given (Monthly Targets)"))
        ycols = ["Year", "Region", "Branch", "GTM", "Activity", "Target", "Basis", "Source"]
        mcols = ["Year", "Month", "Region", "Branch", "GTM", "Activity", "Target", "Basis", "Source"]
        pcols = ["Year", "Month", "Region", "Branch", "GTM", "Product group", "Target", "Basis", "Source"]
        return (pd.DataFrame(y_rows, columns=ycols), pd.DataFrame(m_rows, columns=mcols), pd.DataFrame(p_rows, columns=pcols),
                yearly, tm, region)


    def product_targets_yearly(master, sales, region, issues):
        """'Product Targets' (family x region x year, as given) shared out to the region's branches by each branch's share
        of that product group's sales in the year before. Only products that have a target get rows."""
        p = master.get("Product Targets")
        if p is None:
            return pd.DataFrame()
        raw = p.iloc[:, :4].copy()
        raw.columns = ["Product group", "Region", "Year", "Target"]
        raw["Target"] = num(raw["Target"])
        if raw.duplicated(["Product group", "Region", "Year"]).any():
            issues.add("Product Targets", "'East' appears twice per family (East1 and East2 both written as East)",
                       "added together as East", f"{raw.duplicated(['Product group', 'Region', 'Year'], keep=False).sum()} rows")
        if p.shape[1] > 4:
            issues.add("Product Targets", "a list of family names to the right of the data", "not read")
        unknown = sorted(set(raw["Product group"].dropna()) - set(GROUP_ACTIVITY))
        if unknown:
            issues.add("Product Targets", "families that are not a product group: " + ", ".join(unknown), "kept, but no sales match")
        t = raw.groupby(["Year", "Region", "Product group"])["Target"].sum()
        retail = sales[sales["GTM"] == "Retail"]
        branches = {}
        for b, r in region.items():
            branches.setdefault(r, []).append(b)
        rows, years = [], sorted({y for y, _, _ in t.index})
        for (y, reg, grp), v in t.items():
            bs = branches.get(reg, [])
            base = retail[(retail["Year"] == y - 1) & (retail["Group"] == grp) & retail["Branch"].isin(bs)]
            sh = base.groupby("Branch")["Gross"].sum()
            if sh.sum() <= 0:
                sh = pd.Series(1.0, index=bs)
                how = "Region target shared equally (no sales the year before)"
            else:
                how = f"Region target, shared by the branch's {y - 1} sales of this product"
            sh = sh / sh.sum()
            for b, share in sh.items():
                rows.append((y, reg, b, "Retail", grp, v * share, "Net", how))
        if years:
            issues.add("Product Targets", f"yearly product targets only for {', '.join(map(str, years))}, by region",
                       "shared out to branches by the branch's sales of each product the year before")
        return pd.DataFrame(rows, columns=["Year", "Region", "Branch", "GTM", "Product group", "Target", "Basis", "Source"])


    def project_targets(hs, sales, year, issues):
        """Project yearly targets from the HS file's 'Branch Targets' (branch x activity, as given)."""
        if hs is None or "Branch Targets" not in hs:
            return pd.DataFrame()
        bt = hs["Branch Targets"].copy()
        bad = bt["Target"].notna() & num(bt["Target"]).isna()
        if bad.any():
            issues.add("HS Branch Targets", "targets that are not numbers", "taken as empty",
                       ", ".join(sorted({str(v) for v in bt.loc[bad, "Target"]}))[:200])
        bt["Target"] = num(bt["Target"])
        bt["Year"] = num(bt["Year"])
        rr = bt[(bt["Year"] == year) & (bt["Category"] == "Retail")]
        if len(rr) and rr["Target"].nunique() == 1:
            issues.add("HS Branch Targets", f"every Retail branch has the same {year} target ({rr['Target'].iloc[0]:,.0f})",
                       "replaced by the Retail 'Branch Targets' (Retail and HS now share one Retail target)")
        bt = bt[(bt["Year"] == year) & (bt["Category"] == "Project")]
        dup = bt[bt.duplicated(["Branch", "Helios Code"], keep=False)]
        if len(dup):
            issues.add("HS Branch Targets", "two rows for the same branch and Helios code (HDCOS twice per branch)",
                       "added together as COS", f"{dup['Branch'].nunique()} branches")
        bt["Activity"] = bt["Helios Code"].map(ACTIVITY)
        region = project_regions(sales)
        missing = sorted(set(bt["Branch"]) - set(region))
        if missing:
            issues.add("HS Branch Targets", "Project branches with no Project sales last year", "region left blank",
                       ", ".join(missing))
        rows = [(year, region.get(b), b, "Project", act, v, "Net", "Given (HS Branch Targets)")
                for (b, act), v in bt.groupby(["Branch", "Activity"])["Target"].sum().items()]
        return pd.DataFrame(rows, columns=["Year", "Region", "Branch", "GTM", "Activity", "Target", "Basis", "Source"])


    def project_regions(sales):
        p = sales[sales["GTM"] == "Project"]
        return p.groupby("Branch")["Region"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None).to_dict()


    def project_monthly(com, year, issues, yearly_p=None):
        """Project monthly targets = the HS file's Project commits (branch x activity x month, as given).
        The HS file's Retail commits are not used: Retail monthly targets come from the Retail 'Monthly Targets' sheet,
        so the Retail and HS dashboards show the same Retail numbers."""
        if com is None or com.empty:
            return pd.DataFrame()
        if (com["GTM"] == "Retail").any():
            issues.add("Activity wise Month Target", "Retail commits in the HS file differ from the Retail 'Monthly Targets'",
                       "not used — Retail monthly targets come from the Retail file for both dashboards",
                       f"{(com['GTM'] == 'Retail').sum()} rows")
        p = com[(com["GTM"] == "Project") & com["Activity"].notna()]
        fd = p[p["Activity"] == "FD"]
        if len(fd):
            issues.add("Activity wise Month Target", "Project FD commits (FD is always Retail; values up to 124 Cr a month)",
                       "not used", f"{len(fd)} rows")
        p = p[p["Activity"] != "FD"]
        if yearly_p is not None and len(yearly_p):
            # a month whose commit equals the whole yearly target is the yearly figure pasted in, not a monthly commit
            y = yearly_p.groupby(["Branch", "Activity"])["Target"].sum()
            tot = p.groupby(["Branch", "Month", "Activity"])["Commit"].transform("sum")
            yv = pd.Series([y.get((b, a), np.nan) for b, a in zip(p["Branch"], p["Activity"])], index=p.index)
            pasted = (yv > 0) & ((tot - yv).abs() <= 0.001 * yv)
            if pasted.any():
                months = sorted(p.loc[pasted, "Month"].dt.strftime("%b").unique())
                issues.add("Activity wise Month Target", f"Project commits for {', '.join(months)} equal the whole yearly target",
                           "not used (the yearly figure pasted in as a month)", f"{p.loc[pasted, 'Branch'].nunique()} branches")
                p = p[~pasted]
        if (com["Commit"].round(0) != com["OB Commit"].round(0)).mean() < 0.01:
            issues.add("Activity wise Month Target", "'OB Commit' is the same as 'Commit' on almost every row",
                       "only 'Commit' kept (as the monthly target)")
        g = p.groupby(["Branch", "Month", "Region", "Activity"], dropna=False)["Commit"].sum()
        rows = [(year, pd.Timestamp(m).date(), reg, b, "Project", act, v, "Net", "Given (HS commit)")
                for (b, m, reg, act), v in g.items()]
        return pd.DataFrame(rows, columns=["Year", "Month", "Region", "Branch", "GTM", "Activity", "Target", "Basis", "Source"])


    def commits(hs, year, issues):
        """The HS file's 'Activity wise Month Target' rows for the year (Commit by branch, month, GTM and activity)."""
        if hs is None or "Activity wise Month Target" not in hs:
            return pd.DataFrame()
        c = hs["Activity wise Month Target"].copy()
        c["Month"] = pd.to_datetime(c["Month and year"], errors="coerce")
        c = c[c["Month"].dt.year == year]
        for k in ("Commit", "OB Commit"):
            c[k] = num(c[k])
        c["Activity"] = c["Helios Code"].map(ACTIVITY)
        months = sorted(c.loc[c["Category"] == "Project", "Month"].dt.month.unique())
        if months and len(months) < 12:
            issues.add("Activity wise Month Target", f"Project commits only for {len(months)} months "
                       f"({', '.join(_dt.date(year, m, 1).strftime('%b') for m in months)})", "other months left empty")
        out = c.rename(columns={"Zone": "Region", "Category": "GTM"})
        retail = out["GTM"] == "Retail"
        out.loc[retail, "Branch"] = out.loc[retail, "Branch"].map(canon_branch)
        return out[["Month", "Region", "Branch", "GTM", "Activity", "Commit", "OB Commit"]]


    def partner_targets(qtod, master, sales, year, partner_info, region, issues):
        q = qtod.copy()
        q.columns = [str(c).strip() for c in q.columns]
        q = q[q["Target Year"] == year].copy()
        q["Partner Code"] = q["CC"].map(clean_code)
        qcols = ["Qtr 1 Target", "Qtr 2 Target", "Qtr 3 Target", "Qtr 4 Target"]
        for c in qcols + ["Target 100%", "Target 108%"] + list(FOCUS_FAMILY):
            q[c] = num(q[c])
        text = [c for c in qcols if qtod[c].map(lambda v: isinstance(v, str)).any()]
        if text:
            issues.add("QTOD - ATOD", f"{', '.join(text)} stored as text", "converted to numbers")
        # partner branch: partner master, else the QTOD branch (aliases fixed)
        pt = master.get("Partner Targets")
        pm_branch = {}
        if pt is not None:
            pm_branch = {clean_code(k): canon_branch(v) for k, v in zip(pt["Sold-to Party"], pt["Branch"]) if canon_branch(v)}
        info_branch = {k: canon_branch(v.get("Branch")) for k, v in partner_info.items() if canon_branch(v.get("Branch"))}
        qb = q["Branch"].map(canon_branch)
        q["Branch"] = [info_branch.get(c) or pm_branch.get(c) or b for c, b in zip(q["Partner Code"], qb)]
        diff = (qb != q["Branch"]).sum()
        codes = sorted({str(b) for b in qtod["Branch"].dropna().unique()} - set(region))
        if codes:
            issues.add("QTOD - ATOD", "branch codes not used by the sales data / Branch Targets: " + ", ".join(codes),
                       f"branch taken from the partner master ({diff} partners changed)")
        q["Partner type"] = q["Partner Code"].map(lambda c: partner_info.get(c, {}).get("RD/KRO"))
        if q["RD/ KRO"].isna().all() if "RD/ KRO" in q else True:
            issues.add("QTOD - ATOD", "'RD/ KRO' is empty", "filled from the partner master / sales data")
        q["Region"] = q["Branch"].map(region)
        bad_q = q[(q[qcols].sum(axis=1) - q["Target 100%"]).abs() > 0.01 * q["Target 100%"]]
        if len(bad_q):
            issues.add("QTOD - ATOD", "quarter targets do not add up to 'Target 100%'",
                       "kept as given (mostly partners added during the year)",
                       "; ".join(f"{p} ({c})" for p, c in zip(bad_q["Partner"], bad_q["Partner Code"]))[:500])
        ratio = q["Target 108%"] / q["Target 100%"]
        bad_r = q[(ratio - 1.08).abs() > 0.005]
        if len(bad_r):
            issues.add("QTOD - ATOD", "'Target 108%' is not 108% of 'Target 100%'",
                       "quarter 108% targets = quarter x 1.08; check the yearly 108% figure",
                       "; ".join(f"{p}: {r:.2f}x" for p, r in zip(bad_r["Partner"], ratio[bad_r.index])))
        dn = q[q["Partner"].duplicated(keep=False)]
        if len(dn):
            issues.add("QTOD - ATOD", "same partner name on different partner codes (check for duplicates)", "kept",
                       ", ".join(sorted(dn["Partner"].unique()))[:400])
        rows = []
        base = sales[(sales["Year"] == year - 1) & (sales["GTM"] == "Retail")]
        qsh_b = shares(base.assign(Q=(base["Month"] - 1) // 3 + 1), ["Branch"], "Q")
        for _, r in q.iterrows():
            qt = r[qcols].fillna(0.0).values.astype(float)
            ok = abs(qt.sum() - r["Target 100%"]) <= 0.01 * r["Target 100%"] and qt.sum() > 0
            qs = qt / qt.sum() if ok else lookup_share(qsh_b, r["Branch"], [1, 2, 3, 4], pd.Series(0.25, index=[1, 2, 3, 4])).values
            for i, qn in enumerate(QUARTERS):
                rows.append((year, qn, r["Partner Code"], r["Partner"], r["Branch"], r["Region"], "Retail", r["Partner type"],
                             "All", qt[i], qt[i] * 1.08, "Gross", "Given (108% = quarter x 1.08)"))
                for col, fam in FOCUS_FAMILY.items():
                    v = r[col] if pd.notna(r[col]) else 0.0
                    rows.append((year, qn, r["Partner Code"], r["Partner"], r["Branch"], r["Region"], "Retail", r["Partner type"],
                                 fam, v * qs[i], v * qs[i] * 1.08, "Gross",
                                 "Yearly, by the partner's quarters" if ok else "Yearly, by the branch's quarters last year"))
        cols = ["Year", "Quarter", "Partner Code", "Partner", "Branch", "Region", "GTM", "Partner type", "Family",
                "Target 100%", "Target 108%", "Basis", "Source"]
        return pd.DataFrame(rows, columns=cols), q


    def ssp_targets(master, year, issues):
        s = master.get("SSA Targets")
        if s is None:
            return pd.DataFrame()
        s = s.copy()
        s["SSP (as given)"] = s["SSP"].astype(str).str.strip()
        s["SSP"] = s["SSP (as given)"].replace(SSP_ALIASES)
        ch = s[s["SSP"] != s["SSP (as given)"]]
        if len(ch):
            issues.add("SSA Targets", "SSP names spelt differently from the sales data",
                       "renamed: " + ", ".join(sorted({f"{a} -> {b}" for a, b in zip(ch['SSP (as given)'], ch['SSP'])})))
        s["Branch (as given)"] = s["Branch"]
        s["Branch"] = s["Branch"].map(canon_branch)
        if "old Target" in s and s["old Target"].notna().any():
            issues.add("SSA Targets", "'old Target' column", "kept as 'Previous target' for reference")
        s = s.rename(columns={"old Target": "Previous target"})
        s["Basis"] = "Net"
        return s[["Year", "SSP", "Branch", "Target", "Previous target", "Basis", "SSP (as given)", "Branch (as given)"]]


    def region_targets(master, issues):
        z = master.get("Zone Targets")
        if z is None:
            return pd.DataFrame()
        z = z.rename(columns={"Zone": "Region"})[["Year", "Region", "Target"]].copy()
        z["Basis"] = "Net"
        if z["Region"].astype(str).str.match(r"East\d").any():
            issues.add("Zone Targets", "East is split into East1 / East2 here but is one region (East) in 'Branch Targets'",
                       "kept as given")
        return z


    def overall_and_activity(df):
        """Branch targets → (overall, by activity).
        Overall: one row per branch × GTM (× month), no Activity column — Retail as given (its targets are per
        branch), Project = the sum of its activity targets. By activity: only the rows given per activity
        (Project COS / SPE / WD ...); Retail has no activity targets, so it has no rows there."""
        keys = [c for c in ("Year", "Month", "Region", "Branch", "GTM", "Basis") if c in df.columns]
        act = df[df["Activity"] != "All"].copy()
        whole = df[df["Activity"] == "All"].drop(columns="Activity")
        summed = (act.groupby(keys, dropna=False, as_index=False)["Target"].sum()
                  .assign(Source=lambda d: "Sum of activities (" + d["GTM"] + " targets by activity)"))
        overall = pd.concat([whole, summed], ignore_index=True)[keys[:-1] + ["Target", "Basis", "Source"]]
        return overall, act.reset_index(drop=True)


    def organize(master, qtod, sales, year, hs=None, partner_info=None):
        """master / qtod / hs: {sheet: DataFrame} of the target files.
        Returns ({sheet: DataFrame}, issues DataFrame, {name: Series} of source totals for the Check sheet)."""
        issues = Issues()
        sales = prepare_sales(sales)
        partner_info = partner_info or {}
        yearly_r, monthly_r, prod_m, yearly_given, monthly_given, region = retail_targets(master, year, issues)
        prod_y = product_targets_yearly(master, sales, region, issues)
        yearly_p = project_targets(hs, sales, year, issues)
        com = commits(hs, year, issues)
        if len(com) and len(yearly_p):
            # the commit sheet writes some Project branches in capitals (MUM, PUNE); use the target sheet's spelling
            names = {b.upper(): b for b in yearly_p["Branch"].unique()}
            p_rows = com["GTM"] == "Project"
            fixed = com.loc[p_rows, "Branch"].map(lambda b: names.get(str(b).upper(), b))
            changed = sorted(set(com.loc[p_rows, "Branch"][fixed != com.loc[p_rows, "Branch"]]))
            if changed:
                issues.add("Activity wise Month Target", "Project branch names in capitals: " + ", ".join(changed),
                           "matched to the Project targets' spelling")
            com.loc[p_rows, "Branch"] = fixed
        monthly_p = project_monthly(com, year, issues, yearly_p)
        yearly = pd.concat([yearly_r, yearly_p], ignore_index=True)
        monthly = pd.concat([monthly_r, monthly_p], ignore_index=True)
        yearly, monthly = yearly[yearly["Target"].round(0) != 0], monthly[monthly["Target"].round(0) != 0]
        pt, _ = partner_targets(qtod, master, sales, year, partner_info, region, issues)
        branches = (pd.concat([yearly[["Branch", "Region", "GTM"]], monthly[["Branch", "Region", "GTM"]]])
                    .drop_duplicates(["Branch", "GTM"]).sort_values(["GTM", "Region", "Branch"]))
        branches["Also written as"] = branches["Branch"].map(
            lambda b: ", ".join(sorted(k for k, v in BRANCH_ALIASES.items() if v == b)))
        yearly_o, yearly_a = overall_and_activity(yearly)
        monthly_o, monthly_a = overall_and_activity(monthly)
        tables = {"Branches": branches, "Yearly Targets": yearly_o, "Monthly Targets": monthly_o,
                  "Yearly Targets by Activity": yearly_a, "Monthly Targets by Activity": monthly_a,
                  "Product Targets Monthly": prod_m, "Product Targets Yearly": prod_y,
                  "Partner Targets": pt, "SSP Targets": ssp_targets(master, year, issues),
                  "Region Targets": region_targets(master, issues)}
        for k in tables:
            for c in ("Target", "Target 100%", "Target 108%"):
                if c in tables[k]:
                    tables[k][c] = tables[k][c].astype(float).round(0)
        given = pd.DataFrame({"Yearly target (given)": yearly_given, "Monthly targets (given, all months)": monthly_given})
        return tables, pd.DataFrame(issues), given


    # ====================================================================== workbook
    def write_targets(path, tables, issues, given, year, base_year):
        from openpyxl import Workbook
        from openpyxl.styles import Alignment, Font, PatternFill
        from openpyxl.utils import get_column_letter
        from openpyxl.worksheet.datavalidation import DataValidation
        from openpyxl.worksheet.table import Table, TableStyleInfo

        green = "1F7A4D"
        wb = Workbook()
        ws = wb.active
        ws.title = "Read me"
        notes = [
            (f"Channel Targets {year}", "Retail, HS (Retail + Project) and Wiser targets in one place, one row per target."),
            ("", ""),
            ("Source", "Built from 'Master Targets File', 'QTOD and ATOD file' and the HS targets file. Targets are kept as "
                       "given; column 'Source' says where each number comes from."),
            ("Amounts", "Rupees. 'Basis' says Net or Gross: branch, monthly, product and SSP targets are Net "
                        "(gross = net / 0.83); partner targets are Gross."),
            ("Yearly Targets", "OVERALL — year x branch x GTM, no activity. Retail: 'Branch Targets'; Project: the sum of "
                               "the HS yearly targets by activity. Use for overall target vs sales."),
            ("Monthly Targets", "OVERALL — month x branch x GTM: the monthly commit, no activity. Retail: 'Monthly "
                                "Targets'; Project: the sum of the HS commits by activity. Separate from the yearly target "
                                "— the months need not add up to it, and months not set yet are empty."),
            ("Yearly Targets by Activity", "ACTIVITY-WISE — year x branch x GTM x activity, only where targets are given "
                                           "per activity (Project: COS / SPE / WD ...). Retail targets are per branch, so "
                                           "Retail has no rows here."),
            ("Monthly Targets by Activity", "ACTIVITY-WISE — month x branch x GTM x activity: the HS commits by activity "
                                            "(Project)."),
            ("Product Targets Monthly", "Month x branch x product group, only where a product target is given "
                                        "(Miluz Zeta and Wiser from 'Monthly Targets')."),
            ("Product Targets Yearly", "Year x branch x product group, from 'Product Targets' (region level), shared out to "
                                       "the region's branches by each branch's sales of that product the year before."),
            ("Partner Targets", "Partner x quarter x family ('All' = the partner's total, as in QTOD)."),
            ("SSP Targets", "SSP x branch x year, names as in the sales data."),
            ("Region Targets", "Zone targets as given."),
            ("Issues", "Everything found in the source files and what was done about it."),
            ("Check", "Each sheet against the source file; the 'Difference' columns should be 0."),
            ("Editing", "Add rows at the bottom of a table; keep one row per key; use the dropdowns. Products without a "
                        "target simply have no rows."),
        ]
        ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 22, 120
        for r, (a, b) in enumerate(notes, 1):
            ws.cell(r, 1, a).font = Font(bold=True, size=14 if r == 1 else 11, color=green if r == 1 else "000000")
            ws.cell(r, 2, b).alignment = Alignment(wrap_text=True, vertical="top")

        fmt = {"Month": "mmm-yy", "Target": "#,##0", "Target 100%": "#,##0", "Target 108%": "#,##0", "Commit": "#,##0",
               "OB Commit": "#,##0", "Previous target": "#,##0"}

        def sheet(name, df, widths=None):
            w = wb.create_sheet(name)
            df = df.copy()
            if "Month" in df:
                df["Month"] = pd.to_datetime(df["Month"]).dt.date
            cols = list(df.columns)
            for j, c in enumerate(cols, 1):
                cell = w.cell(1, j, c)
                cell.font, cell.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor=green)
                cell.alignment = Alignment(wrap_text=True, vertical="center")
                w.column_dimensions[get_column_letter(j)].width = (widths or {}).get(c, max(10, min(40, len(c) + 4)))
            for row in df.itertuples(index=False):
                w.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
            w.row_dimensions[1].height = 32
            w.freeze_panes = "A2"
            n = max(2, len(df) + 1)
            t = Table(displayName=re.sub(r"\W", "", name), ref=f"A1:{get_column_letter(len(cols))}{n}")
            t.tableStyleInfo = TableStyleInfo(name="TableStyleLight9", showRowStripes=True)
            w.add_table(t)
            for j, c in enumerate(cols, 1):
                if c in fmt:
                    for r in range(2, n + 1):
                        w.cell(r, j).number_format = fmt[c]
            return w, cols

        lists = {"GTM": ["Retail", "Project"], "Activity": ACTIVITIES, "Product group": [g for g, _ in GROUPS],
                 "Family": ["All", "Miluz Zeta", "Wiser"], "Basis": ["Net", "Gross"], "Quarter": list(QUARTERS)}

        def dropdowns(w, cols):
            for c, vals in lists.items():
                if c in cols:
                    letter = get_column_letter(cols.index(c) + 1)
                    dv = DataValidation(type="list", formula1='"' + ",".join(vals) + '"', allow_blank=True)
                    dv.add(f"{letter}2:{letter}50000")
                    w.add_data_validation(dv)

        order = ["Branches", "Yearly Targets", "Monthly Targets", "Yearly Targets by Activity", "Monthly Targets by Activity",
                 "Product Targets Monthly", "Product Targets Yearly",
                 "Partner Targets", "SSP Targets", "Region Targets"]
        for name in order:
            df = tables.get(name)
            if df is None or df.empty:
                continue
            w, cols = sheet(name, df, {"Source": 46, "Partner": 30, "Problem": 60})
            dropdowns(w, cols)
        if len(issues):
            w, _ = sheet("Issues", issues, {"Source sheet": 24, "Problem": 70, "What was done": 60, "Rows / values": 80})
            for row in w.iter_rows(min_row=2):
                for c in row:
                    c.alignment = Alignment(wrap_text=True, vertical="top")
        # check: per Retail branch, each sheet against the source file
        yt, mt, pm, pt = (tables[k] for k in ("Yearly Targets", "Monthly Targets", "Product Targets Monthly", "Partner Targets"))
        chk = given.copy()
        chk["Yearly Targets sheet"] = yt[yt["GTM"] == "Retail"].groupby("Branch")["Target"].sum()
        chk["Difference (yearly)"] = (chk["Yearly Targets sheet"] - chk["Yearly target (given)"]).round(0)
        chk["Monthly Targets sheet"] = mt[mt["GTM"] == "Retail"].groupby("Branch")["Target"].sum()
        chk["Difference (monthly)"] = (chk["Monthly Targets sheet"] - chk["Monthly targets (given, all months)"]).round(0)
        chk["Months set"] = mt[mt["GTM"] == "Retail"].groupby("Branch")["Month"].nunique()
        chk["Months as % of yearly (info)"] = chk["Monthly Targets sheet"] / chk["Yearly target (given)"]
        for grp in FOCUS.values():
            chk[f"{grp} (monthly product)"] = pm[pm["Product group"] == grp].groupby("Branch")["Target"].sum()
        chk["Partner targets 100% (gross)"] = pt[pt["Family"] == "All"].groupby("Branch")["Target 100%"].sum()
        chk = chk.reset_index().rename(columns={"index": "Branch"})
        w, cols = sheet("Check", chk, {"Branch": 14})
        for j, c in enumerate(cols, 1):
            f = "0%" if c.startswith("Months as") else ("0" if c == "Months set" else ("#,##0" if c != "Branch" else None))
            if f:
                for rr in range(2, len(chk) + 2):
                    w.cell(rr, j).number_format = f
        wb.move_sheet("Check", offset=-(len(wb.sheetnames) - 2))
        if "Issues" in wb.sheetnames:
            wb.move_sheet("Issues", offset=-(wb.sheetnames.index("Issues") - 2))
        for w in wb.worksheets:
            w.sheet_properties.tabColor = green if w.title in ("Read me", "Check", "Issues") else "8FBC9F"
        wb.save(path)


    def write_contribution(path, tables):
        with pd.ExcelWriter(path, engine="openpyxl") as xw:
            for name, df in tables.items():
                df.to_excel(xw, sheet_name=name[:31], index=False)

    return types.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

TS = _target_split_tools()
print("✅ Target rules ready")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 5 · READ THE INPUT FILES (cached locally; re-read only when a file changes)
# ════════════════════════════════════════════════════════════════════
import hashlib


def _file_bytes(key):
    sp_path, local_name = INPUTS[key]
    if SOURCE == "local":
        with open(os.path.join(LOCAL_DIR, local_name), "rb") as fh:
            return fh.read()
    return sp.download(sp_path).getvalue()


def read_book(key, sheet_name=None):
    """All sheets (or `sheet_name`) of an input file as {sheet: DataFrame}, cached as pickle by content."""
    data = _file_bytes(key)
    digest = hashlib.md5(data).hexdigest()[:12]
    cache = os.path.join(CACHE_DIR, f"{key}_{digest}.pkl")
    if os.path.exists(cache):
        return pd.read_pickle(cache)
    t0 = time.time()
    try:
        book = pd.read_excel(io.BytesIO(data), sheet_name=sheet_name, engine="calamine")
    except Exception:
        book = pd.read_excel(io.BytesIO(data), sheet_name=sheet_name)
    if not isinstance(book, dict):
        book = {sheet_name: book}
    for name, df in book.items():
        df.columns = [str(c).strip() for c in df.columns]
    for old in [f for f in os.listdir(CACHE_DIR) if f.startswith(f"{key}_")]:
        os.remove(os.path.join(CACHE_DIR, old))
    pd.to_pickle(book, cache)
    print(f"   📥 {INPUTS[key][1]}: {sum(len(d) for d in book.values()):,} rows in {len(book)} sheet(s), {time.time() - t0:.0f}s")
    return book


def _stack(book, prefix):
    parts = [df for name, df in book.items() if str(name).lower().startswith(prefix.lower()) and len(df)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 6 · PREPARE — one table per data type with the same columns for Retail and HS
#
#   Date · GTM (Retail / Project) · Branch · Region · Activity · Product Group · Family
#   Partner · Partner name · City · Partner type (RD / KRO / Project) · Tier · Gross · Qty
# Branch: the target branch (Retail: partner's branch; Project: sales-office branch);
# Region: the Branch Targets zone for Retail branches, else the input file's region.
# ════════════════════════════════════════════════════════════════════
def _clean_code(s):
    return s.astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True)


def load_masters():
    mm = read_book("material", "Sheet1")["Sheet1"]
    family = dict(zip(_clean_code(mm["Material"]), mm["Family Actual"]))
    d = read_book("dictionary")
    pm = d.get("Partner Master Data", pd.DataFrame())
    geo = d.get("Geography mapping", pd.DataFrame())
    ssa = d.get("SSA Mapping", pd.DataFrame())
    if len(pm):
        pm = pm.copy()
        pm["Partner"] = _clean_code(pm["Sold-to Party"])
        pm["City"] = pm["City"].astype(str).str.strip()
        if len(geo):
            g = geo[["City", "Branch"]].copy()
            g["City"] = g["City"].astype(str).str.strip()
            pm = pm.merge(g.drop_duplicates("City").rename(columns={"Branch": "Geo branch"}), on="City", how="left")
        pm = pm.drop_duplicates("Partner").set_index("Partner")
    ssa_map = {}
    if len(ssa) and {"Branch", "SSA"} <= set(ssa.columns):
        ssa_map = dict(zip(ssa["Branch"].astype(str).str.strip().str.upper(), ssa["SSA"]))
    return {"family": family, "partners": pm, "ssa": ssa_map}


def standardise(df, kind, M, region_of, gtm=None):
    """Input rows → the standard columns. kind = 'sales' | 'ob' | 'backlog'."""
    if df is None or df.empty:
        return pd.DataFrame(columns=["Date", "GTM", "Branch", "Region", "Activity", "Product Group", "Family", "Partner",
                                     "Partner name", "City", "Partner type", "Tier", "Gross", "Qty", "Order date"])
    out = pd.DataFrame(index=df.index)
    date_col = {"sales": "InvoiceDate", "ob": "Calendar Day", "backlog": None}[kind]
    out["Date"] = pd.to_datetime(df[date_col], errors="coerce") if date_col in df else pd.NaT
    out["Order date"] = pd.to_datetime(df["OrderCreationDate"], errors="coerce") if "OrderCreationDate" in df else pd.NaT
    out["GTM"] = gtm if gtm else (df["Target GTM"] if "Target GTM" in df else df.get("Classification", "Retail"))
    partner = _clean_code(df["SoldToPartyID"])
    out["Partner"] = partner
    pm = M["partners"]
    retail = out["GTM"] == "Retail"
    # branch: the target key column when the notebook wrote it, else partner master (Retail) / Branch (HS file)
    if "Target Branch" in df:
        branch = df["Target Branch"]
    else:
        branch = df["Branch"] if "Branch" in df else pd.Series(np.nan, index=df.index)
        if len(pm) and "Geo branch" in pm:
            branch = branch.where(~retail, partner.map(pm["Geo branch"]))
    names = {b: TS.canon_branch(b) for b in pd.Series(branch[retail]).dropna().unique()}
    out["Branch"] = branch.where(~retail, branch.map(names)).fillna("Unmapped")
    reg_in = df["Region"] if "Region" in df else pd.Series(np.nan, index=df.index)
    out["Region"] = out["Branch"].map(region_of).where(retail, reg_in).fillna(reg_in).fillna("Unmapped")
    helios = df["HeliosCode"].astype(str).str.strip().str.upper()
    out["Helios"] = helios
    out["Activity"] = helios.map(TS.ACTIVITY).fillna("Other")
    fam = _clean_code(df["MaterialCode"]).map(M["family"])
    out["Family"] = fam.fillna("Unmapped")
    pairs = {(a, f): TS.product_group(a, f) for a, f in set(zip(out["Activity"], fam))}
    out["Product Group"] = [pairs[(a, f)] or "Other" for a, f in zip(out["Activity"], fam)]
    out["Partner name"] = partner.map(pm["Patner"]) if len(pm) and "Patner" in pm else np.nan
    out["Partner name"] = out["Partner name"].fillna(df.get("SoldToPartyName")).fillna(partner)
    out["City"] = (partner.map(pm["City"]) if len(pm) else np.nan)
    out["City"] = out["City"].where(retail).fillna(df.get("PartnerCity")).astype(str).str.strip().str.title()
    l3 = df["PricingClassL3"].astype(str).str.strip() if "PricingClassL3" in df else pd.Series("", index=df.index)
    ptype = l3.map({"ERW": "RD", "EKR": "KRO"})
    if len(pm) and "RD/KRO" in pm:
        ptype = partner.map(pm["RD/KRO"].astype(str).str.strip().str.upper().map({"RD": "RD", "KRO": "KRO"})).fillna(ptype)
    out["Partner type"] = np.where(~retail, "Project", ptype.fillna("Other Retail"))
    tier_col = next((c for c in pm.columns if "tier" in c.lower()), None) if len(pm) else None
    out["Tier"] = partner.map(pm[tier_col]).fillna("Unmapped") if tier_col else "Unmapped"
    out["Gross"] = pd.to_numeric(df["Gross Sales"], errors="coerce").fillna(0.0)
    out["Qty"] = pd.to_numeric(df.get("BillingQuantity", 0), errors="coerce").fillna(0.0)
    out["Material"] = df.get("MaterialCode")
    out["Order"] = df.get("SalesOrderHeader")
    return out


def load_data():
    M = load_masters()
    book = read_book("targets")
    tmp = TS.Issues()
    _, _, _, _, _, region_of = TS.retail_targets(book, pd.Timestamp.today().year, tmp)
    r = read_book("retail")
    h = read_book("hs")
    D = {
        "retail_sales": standardise(_stack(r, "Sales"), "sales", M, region_of, "Retail"),
        "retail_ob": standardise(_stack(r, "OB"), "ob", M, region_of, "Retail"),
        "retail_bl": standardise(_stack(r, "Backlog"), "backlog", M, region_of, "Retail"),
        "hs_sales": standardise(_stack(h, "Sales"), "sales", M, region_of),
        "hs_ob": standardise(_stack(h, "OB"), "ob", M, region_of),
        "hs_bl": standardise(_stack(h, "Backlog"), "backlog", M, region_of),
    }
    D["masters"] = M
    D["books"] = {"targets": book, "qtod": next(iter(read_book("qtod").values())), "hs_targets": read_book("hs_targets")}
    for k in ("retail_sales", "hs_sales"):
        print(f"   {k}: {len(D[k]):,} rows, {D[k]['Date'].min():%d %b %Y} – {D[k]['Date'].max():%d %b %Y}")
    return D

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 7 · PERIOD, VALUES AND TARGETS
#
# Net = Gross × (1 − rebate of the row's GTM and year) — REBATES in cell 1.
# Targets in the target files are net: the Gross files show them grossed up with the same rebate
# (target ÷ (1 − rebate)); QTOD partner targets are gross and are netted the same way for the Net files.
# ════════════════════════════════════════════════════════════════════
def period(D):
    last = D["retail_sales"]["Date"].max()
    as_of = pd.Timestamp(AS_OF) if AS_OF else last
    as_of = min(as_of, max(last, D["hs_sales"]["Date"].max())).normalize()
    y = as_of.year

    def ly(d):
        try:
            return d.replace(year=d.year - 1)
        except ValueError:
            return d.replace(year=d.year - 1, day=28)
    return {"as_of": as_of, "year": y, "m_start": as_of.replace(day=1), "y_start": pd.Timestamp(y, 1, 1),
            "ly_as_of": ly(as_of), "ly_m_start": ly(as_of.replace(day=1)), "ly_y_start": pd.Timestamp(y - 1, 1, 1),
            "ly_end": pd.Timestamp(y - 1, 12, 31), "months_left": 12 - as_of.month + (0 if as_of.is_month_end else 1),
            "label": as_of.strftime("%d %b %Y"), "mon": as_of.strftime("%b")}


def rebate(gtm, year):
    table = REBATES.get(gtm, REBATES["Retail"])
    return table.get(int(year), table[max(table)]) if pd.notna(year) else table[max(table)]


def with_values(df, basis, P):
    """Copy of df with 'Value' = Gross or Net (rebate by GTM and year; backlog at this year's rate)."""
    out = df.copy()
    if basis == "Gross" or out.empty:
        out["Value"] = out["Gross"]
        return out
    year = out["Date"].dt.year.fillna(P["year"]).astype(int)
    rates = {(g, yy): rebate(g, yy) for g, yy in set(zip(out["GTM"], year))}
    out["Value"] = out["Gross"] * (1 - pd.Series([rates[k] for k in zip(out["GTM"], year)], index=out.index))
    return out


def targets(D, P):
    """All targets of the year (net, as in the files)."""
    y, issues = P["year"], TS.Issues()
    book, hs_book = D["books"]["targets"], D["books"]["hs_targets"]
    _, _, prod_m, yearly, _, region_of = TS.retail_targets(book, y, issues)
    mt = book["Monthly Targets"].iloc[:, :4].copy()
    mt.columns = ["Zone", "Branch", "Month", "Target"]
    mt["Branch"] = mt["Branch"].map(TS.canon_branch)
    mt["Month"] = pd.to_datetime(mt["Month"], errors="coerce")
    mt["Target"] = pd.to_numeric(mt["Target"], errors="coerce")
    monthly = mt[mt["Month"].dt.year == y].dropna(subset=["Target"]).groupby(["Branch", "Month"])["Target"].sum()
    hs = D["hs_sales"]
    sales = TS.prepare_sales(pd.DataFrame({"Year": hs["Date"].dt.year, "Month": hs["Date"].dt.month, "GTM": hs["GTM"],
                                           "Branch": hs["Branch"], "Region": hs["Region"], "HeliosCode": hs["Helios"],
                                           "Family": hs["Family"], "Partner": hs["Partner"], "Gross": hs["Gross"]})
                             .dropna(subset=["Year"]))
    py = TS.project_targets(hs_book, sales, y, issues)
    com = TS.commits(hs_book, y, issues)
    if len(com) and len(py):
        names = {b.upper(): b for b in py["Branch"].unique()}
        p = com["GTM"] == "Project"
        com.loc[p, "Branch"] = com.loc[p, "Branch"].map(lambda b: names.get(str(b).upper(), b))
    pm_ = TS.project_monthly(com, y, issues, py)
    pm = D["masters"]["partners"]
    info = {k: {"Branch": TS.canon_branch(v), "RD/KRO": t} for k, v, t in
            zip(pm.index, pm.get("Geo branch", pd.Series(index=pm.index, dtype=object)), pm.get("RD/KRO", pd.Series(index=pm.index, dtype=object)))} if len(pm) else {}
    pt, _ = TS.partner_targets(D["books"]["qtod"], book, sales, y, info, region_of, issues)
    return {"yearly": yearly, "monthly": monthly, "prod_m": prod_m, "region_of": region_of,
            "project_yearly": py, "commits": com, "project_monthly": pm_, "partners": pt,
            "ssp": TS.ssp_targets(book, y, issues), "issues": pd.DataFrame(issues)}


def to_basis(v, gtm, basis, year):
    """Net target → the report's basis."""
    return v if basis == "Net" else v / (1 - rebate(gtm, year))

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 8 · ANALYSIS BUILDING BLOCKS
# ════════════════════════════════════════════════════════════════════
M_ = f" ({UNIT_LABEL})"


def _sum(df, start=None, end=None, col="Value"):
    if df is None or df.empty:
        return 0.0
    if start is None:
        return df[col].sum()
    d = df["Date"]
    return df.loc[(d >= start) & (d <= end), col].sum()


def _pct(a, b):
    return a / b - 1 if b else np.nan


def _ach(a, t):
    return a / t if pd.notna(t) and t else np.nan


def scorecard(df, keys, P, ob=None, bl=None, month_target=None, year_target=None, qty=False):
    """MTD / YTD / last year by `keys`, growth, targets (already on the report basis), YTD OB, open backlog."""
    def by(start, end, col="Value", frame=df):
        if frame is None or frame.empty:
            return pd.Series(dtype=float)
        d = frame["Date"]
        return frame[(d >= start) & (d <= end)].groupby(keys)[col].sum()
    t = pd.DataFrame({
        "MTD Sales": by(P["m_start"], P["as_of"]), "MTD Sales LY": by(P["ly_m_start"], P["ly_as_of"]),
        "YTD Sales": by(P["y_start"], P["as_of"]), "YTD Sales LY": by(P["ly_y_start"], P["ly_as_of"]),
        f"FY {P['year'] - 1} Sales": by(P["ly_y_start"], P["ly_end"])}).fillna(0.0)
    t["MTD Growth %"] = t["MTD Sales"] / t["MTD Sales LY"].where(t["MTD Sales LY"] > 0) - 1
    t["YTD Growth %"] = t["YTD Sales"] / t["YTD Sales LY"].where(t["YTD Sales LY"] > 0) - 1
    if month_target is not None:
        t.insert(0, "Month Target", pd.to_numeric(month_target.reindex(t.index), errors="coerce"))
        t["Month Ach %"] = t["MTD Sales"] / t["Month Target"].where(t["Month Target"] > 0)
    if year_target is not None:
        t["Yearly Target"] = pd.to_numeric(year_target.reindex(t.index), errors="coerce")
        t["YTD Ach %"] = t["YTD Sales"] / t["Yearly Target"].where(t["Yearly Target"] > 0)
        t["Balance to target"] = (t["Yearly Target"] - t["YTD Sales"]).clip(lower=0)
        t["Needed per month"] = t["Balance to target"] / max(P["months_left"], 1)
    if qty:
        t["YTD Qty"] = by(P["y_start"], P["as_of"], "Qty").reindex(t.index).fillna(0)
        t["YTD Qty LY"] = by(P["ly_y_start"], P["ly_as_of"], "Qty").reindex(t.index).fillna(0)
        t["Qty Growth %"] = t["YTD Qty"] / t["YTD Qty LY"].where(t["YTD Qty LY"] > 0) - 1
    if ob is not None and len(ob):
        t["YTD OB"] = by(P["y_start"], P["as_of"], frame=ob).reindex(t.index).fillna(0.0)
    if bl is not None and len(bl):
        t["Open Backlog"] = bl.groupby(keys)["Value"].sum().reindex(t.index).fillna(0.0)
    return t


def with_total(t, label="Total"):
    """Adds a total row; ratios are recomputed from the totals."""
    if t.empty:
        return t
    s = t.select_dtypes("number").sum(min_count=1)
    for pct_col, a, b in (("MTD Growth %", "MTD Sales", "MTD Sales LY"), ("YTD Growth %", "YTD Sales", "YTD Sales LY"),
                          ("Qty Growth %", "YTD Qty", "YTD Qty LY")):
        if pct_col in t:
            s[pct_col] = s[a] / s[b] - 1 if s.get(b) else np.nan
    for pct_col, a, b in (("Month Ach %", "MTD Sales", "Month Target"), ("YTD Ach %", "YTD Sales", "Yearly Target")):
        if pct_col in t:
            s[pct_col] = s[a] / s[b] if pd.notna(s.get(b)) and s.get(b) else np.nan
    if "Needed per month" in t and "Balance to target" in t:
        s["Needed per month"] = s["Balance to target"] / max(PERIOD["months_left"], 1)
    row = pd.DataFrame([s], index=pd.Index([label] if t.index.nlevels == 1 else [tuple([label] + [""] * (t.index.nlevels - 1))],
                                           name=t.index.name))
    if t.index.nlevels > 1:
        row.index = pd.MultiIndex.from_tuples(row.index, names=t.index.names)
    return pd.concat([t, row])


def month_trend(df, P, targets_by_month=None, year_target=None):
    """Each month of the year: sales this year, last year, the year before, growth; targets and YTD progress."""
    y = P["year"]
    rows, cum = [], 0.0
    for m in range(1, 13):
        a = pd.Timestamp(y, m, 1)
        b = a + pd.offsets.MonthEnd(0)
        cur = _sum(df, a, min(b, P["as_of"])) if a <= P["as_of"] else np.nan
        ly = _sum(df, a.replace(year=y - 1), b.replace(year=y - 1))
        ly2 = _sum(df, a.replace(year=y - 2), (a.replace(year=y - 2) + pd.offsets.MonthEnd(0)))
        cum = cum + (cur if pd.notna(cur) else 0)
        r = {"Month": a.strftime("%b"), f"Sales {y}": cur, f"Sales {y - 1}": ly, f"Sales {y - 2}": ly2,
             "Growth % vs LY": _pct(cur, ly) if pd.notna(cur) else np.nan}
        if targets_by_month is not None:
            t = targets_by_month.get(a, np.nan)
            r["Month Target"] = t
            r["Month Ach %"] = _ach(cur, t)
        if year_target is not None:
            r["YTD Sales"] = cum if pd.notna(cur) else np.nan
            r["YTD Ach % of yearly target"] = _ach(cum, year_target) if pd.notna(cur) else np.nan
            r["Year elapsed %"] = m / 12
        rows.append(r)
    return pd.DataFrame(rows).set_index("Month")


def crosstab(df, rows, cols, P, col="Value"):
    """rows × cols: YTD this year, YTD last year and growth, side by side."""
    def piv(start, end):
        d = df[(df["Date"] >= start) & (df["Date"] <= end)]
        return d.assign(**{col: d[col].astype(float)}).pivot_table(index=rows, columns=cols, values=col, aggfunc="sum", fill_value=0.0)
    ty, ly = piv(P["y_start"], P["as_of"]), piv(P["ly_y_start"], P["ly_as_of"])
    keys = sorted(set(ty.columns) | set(ly.columns), key=str)
    ty, ly = ty.reindex(columns=keys, fill_value=0.0), ly.reindex(index=ty.index.union(ly.index), columns=keys, fill_value=0.0)
    ty = ty.reindex(ly.index, fill_value=0.0)
    out = {}
    for k in keys:
        out[(k, "This year")] = ty[k]
        out[(k, "Last year")] = ly[k]
        out[(k, "Growth %")] = ty[k] / ly[k].where(ly[k] > 0) - 1
    o = pd.DataFrame(out)
    o.columns = [f"{a} — {b}" + (M_ if b != "Growth %" and col == "Value" else "") for a, b in o.columns]
    return o


def partner_table(df, ob, bl, P, extra=None):
    """One row per partner: who, where, sales, growth, growth band, OB, backlog, last billing."""
    t = scorecard(df, ["Partner"], P, ob, bl)
    info = df.sort_values("Date").groupby("Partner").agg(**{"Partner name": ("Partner name", "last"), "Branch": ("Branch", "last"),
                                                          "Region": ("Region", "last"), "City": ("City", "last"),
                                                          "Partner type": ("Partner type", "last"), "Tier": ("Tier", "last"),
                                                          "GTM": ("GTM", "last"), "Last billing": ("Date", "max")})
    t = info.join(t, how="right")
    yl, yy = t["YTD Sales LY"], t["YTD Sales"]
    t["Growth band"] = np.select(
        [(yl <= 0) & (yy > 0), yy <= 0, t["YTD Growth %"] < -0.2, t["YTD Growth %"] < 0, t["YTD Growth %"] < 0.2,
         t["YTD Growth %"] < 0.5], ["New / no sales last year", "No sales this year", "Decline > 20%", "Decline 0–20%",
                                     "Growth 0–20%", "Growth 20–50%"], default="Growth > 50%")
    if extra is not None:
        t = t.join(extra, how="left")
    return t.sort_values("YTD Sales", ascending=False)


def policy_categories(df, P):
    """Partner policy categories as the Partner Analysis policy tracker (needs 3 years of Retail sales)."""
    y = P["year"]
    s = df[df["Date"] <= P["as_of"]]
    sm = s.groupby(["Partner", s["Date"].dt.year])["Gross"].sum().unstack(fill_value=0.0)
    g = lambda yy: sm[yy] if yy in sm.columns else pd.Series(0.0, index=sm.index)
    first_ly = s[s["Date"].dt.year == y - 1].groupby("Partner")["Date"].min().reindex(sm.index)
    new_ty = (g(y) > 0) & (g(y - 1) == 0) & (g(y - 2) == 0)
    new_ly = (g(y - 1) > 0) & (g(y - 2) == 0) & (g(y - 3) == 0)
    q4 = new_ly & first_ly.dt.month.isin([10, 11, 12])
    big = g(y - 1) >= 500000
    cat = pd.Series(np.select([new_ty, q4, new_ly & ~q4 & big, new_ly & ~q4 & ~big],
                              [f"New Partner {y}", f"Q4'{y - 1} Onboarded", f">5L {y - 1} Onboarded (Handholding)",
                               f"<5L {y - 1} Onboarded"], default="Old Partner"), index=sm.index)
    cat[cat.index.isin({str(x) for x in MANUAL_OLD_PARTNER_IDS})] = "Old Partner"
    order = [f"New Partner {y}", f"Q4'{y - 1} Onboarded", f"<5L {y - 1} Onboarded", f">5L {y - 1} Onboarded (Handholding)",
             "Old Partner"]
    bucket = cat.map(lambda c: "New partner" if c in order[:3] else "Handholding" if c == order[3] else "Old partner")
    return cat, bucket, order


def new_geo(df, P):
    """Cities with sales this year and none in the two years before."""
    y = P["year"]
    s = df[(df["Date"] <= P["as_of"]) & df["City"].notna() & (df["City"] != "Nan")]
    cm = s.groupby(["City", s["Date"].dt.year])["Gross"].sum().unstack(fill_value=0.0)
    g = lambda yy: cm[yy] if yy in cm.columns else pd.Series(0.0, index=cm.index)
    cities = cm.index[(g(y) > 0) & (g(y - 1) == 0) & (g(y - 2) == 0)]
    d = s[s["City"].isin(cities) & (s["Date"].dt.year == y)]
    return (d.groupby("City").agg(**{"Region": ("Region", "first"), "Branch": ("Branch", "first"),
                                     "Partners": ("Partner", "nunique"), "First sale": ("Date", "min"), "YTD Sales": ("Value", "sum")})
            .sort_values("YTD Sales", ascending=False))


def backlog_ageing(bl, keys, P):
    if bl is None or bl.empty:
        return pd.DataFrame()
    age = (P["as_of"] - bl["Order date"]).dt.days
    b = bl.assign(Band=pd.cut(age, [-1e9, 30, BACKLOG_OLD_DAYS, 90, 1e9],
                              labels=["0–30 days", f"31–{BACKLOG_OLD_DAYS} days", f"{BACKLOG_OLD_DAYS + 1}–90 days", "Over 90 days"]))
    t = b.pivot_table(index=keys, columns="Band", values="Value", aggfunc="sum", fill_value=0.0, observed=False)
    t.columns = [f"{c}{M_}" for c in t.columns]
    t[f"Open Backlog{M_}"] = t.sum(axis=1)
    t[f"Older than {BACKLOG_OLD_DAYS} days %"] = (b[age > BACKLOG_OLD_DAYS].groupby(keys)["Value"].sum()
                                                  .reindex(t.index).fillna(0) / t[f"Open Backlog{M_}"].where(t[f"Open Backlog{M_}"] > 0))
    return t.sort_values(f"Open Backlog{M_}", ascending=False)


def kpis(rows):
    """(measure, value, note) → a table with the value as text: Mn with 1 decimal, % or a count."""
    def show(measure, v):
        if v is None or (isinstance(v, float) and not np.isfinite(v)):
            return "–"
        if "%" in measure:
            return f"{v:.1%}"
        if measure.endswith(M_):
            return f"{v:,.1f}"
        return f"{v:,.0f}"
    return pd.DataFrame([(m, show(m, v), n) for m, v, n in rows], columns=["Measure", "Value", "Note"]).set_index("Measure")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 9 · WORKBOOKS
# ════════════════════════════════════════════════════════════════════
MONEY_WORDS = ("Sales", "Target", "OB", "Backlog", "Commit", "Needed", "Balance", "Value")


def _for_excel(df):
    """Money columns ÷ UNIT with '(Mn)' in the name; the index as columns; ±inf → blank."""
    out = df.copy()
    ren = {}
    for c in out.columns:
        name = str(c)
        if not pd.api.types.is_numeric_dtype(out[c]) or pd.api.types.is_bool_dtype(out[c]):
            continue
        if name.endswith(M_):
            out[c] = out[c] / UNIT
        elif "%" not in name and not any(w in name for w in ("Qty", "Partners", "partners", "Lines")) \
                and any(w in name for w in MONEY_WORDS):
            out[c] = out[c] / UNIT
            ren[c] = f"{name}{M_}"
    out = out.rename(columns=ren).replace([np.inf, -np.inf], np.nan)
    if not isinstance(out.index, pd.RangeIndex):
        out = out.reset_index()
    return out


def write_book(filename, sheets, P):
    """sheets: list of (sheet name, title, DataFrame, note)."""
    def build(writer):
        for name, title, df, note in sheets:
            if df is None or len(df) == 0:
                continue
            out = _for_excel(df)
            kinds = {c: "int" for c in out.columns if any(w in str(c) for w in ("Qty", "Partners", "Lines", "Months", "Cities"))
                     and pd.api.types.is_numeric_dtype(out[c])}
            kinds.update({c: "mn" for c in out.columns if str(c).endswith(M_)})
            kinds.update({c: "pct" for c in out.columns if "%" in str(c)})
            write_table(writer, out, name[:31], title=title, subtitle=note, tab=THEME["tab_summary"],
                        freeze_cols=min(2, len(out.columns) - 1), kinds=kinds)
    return save_report(build, filename, local_dir=OUTPUT_DIR, sp_folder=SP_OUTPUT_FOLDER if UPLOAD_TO_SHAREPOINT else None)


def about_sheet(scope, basis, P, T):
    rows = [("Report", scope), ("Values", f"{basis} sales, INR {UNIT_LABEL}"),
            ("Period", f"YTD 1 Jan – {P['label']}; MTD {P['m_start']:%d %b} – {P['label']}; last year = the same days a year earlier"),
            ("Rebates (Net)", "; ".join(f"{g}: " + ", ".join(f"{y} {r:.1%}" for y, r in sorted(v.items(), reverse=True))
                                         for g, v in REBATES.items())),
            ("Targets", "Branch yearly and monthly targets, product (Zeta / Wiser) and SSP targets are net in the target files; "
                        + ("the Gross file shows them ÷ (1 − rebate)." if basis == "Gross" else "shown as they are.")
                        + " QTOD partner targets are gross" + (" (netted with the Retail rebate here)." if basis == "Net" else ".")),
            ("Created", datetime.now().strftime("%d %b %Y %H:%M"))]
    t = pd.DataFrame(rows, columns=["Item", "Detail"]).set_index("Item")
    return t


def retail_book(D, T, P, basis):
    y, mon = P["year"], P["mon"]
    S, O, B = (with_values(D[k], basis, P) for k in ("retail_sales", "retail_ob", "retail_bl"))
    conv = lambda s: s.map(lambda v: to_basis(v, "Retail", basis, y)) if len(s) else s
    yt = conv(T["yearly"])
    mt_all = T["monthly"].map(lambda v: to_basis(v, "Retail", basis, y))
    m_key = P["m_start"]
    mt = mt_all.xs(m_key, level="Month") if m_key in mt_all.index.get_level_values(1) else pd.Series(dtype=float)
    region_of = T["region_of"]
    rg_t = lambda s: s.groupby(s.index.map(lambda b: region_of.get(b, "Unmapped"))).sum()
    month_tot = mt_all.groupby(level="Month").sum().to_dict()

    # --- partners and policy
    cat, bucket, order = policy_categories(S, P)
    pt = T["partners"]
    pt = pt[pt["Family"] == "All"] if len(pt) else pt
    p_rate = 1.0 if basis == "Gross" else (1 - rebate("Retail", y))
    q_now = (P["as_of"].month - 1) // 3 + 1
    if len(pt):
        tq = (pt["Target 100%"] * p_rate).groupby([pt["Partner Code"], pt["Quarter"]]).sum().unstack(fill_value=0.0)
        q_start = pd.Timestamp(y, 3 * (q_now - 1) + 1, 1)
        q_sales = S[(S["Date"] >= q_start) & (S["Date"] <= P["as_of"])].groupby("Partner")["Value"].sum()
        extra = pd.DataFrame({"Annual Target (QTOD file)": tq.sum(axis=1), f"Q{q_now} Target": tq.get(f"Q{q_now}"),
                              f"Q{q_now} Sales": q_sales})
    else:
        extra = pd.DataFrame()
    extra = extra.reindex(extra.index.union(cat.index)) if len(extra) else pd.DataFrame(index=cat.index)
    extra["Policy category"] = cat
    extra["Policy group"] = bucket
    flat = extra["Policy group"].eq("New partner") & extra.get("Annual Target (QTOD file)", pd.Series(dtype=float)).reindex(extra.index).isna()
    extra["Assigned Target (policy)"] = extra.get("Annual Target (QTOD file)").where(~flat, FLAT_NEW_PARTNER_TARGET * p_rate) \
        if "Annual Target (QTOD file)" in extra else np.where(extra["Policy group"].eq("New partner"), FLAT_NEW_PARTNER_TARGET * p_rate, np.nan)
    parts = partner_table(S, O, B, P, extra)
    parts["ATOD Ach %"] = parts["YTD Sales"] / parts["Assigned Target (policy)"].where(parts["Assigned Target (policy)"] > 0)
    if f"Q{q_now} Target" in parts:
        parts[f"Q{q_now} QTOD Ach %"] = parts[f"Q{q_now} Sales"] / parts[f"Q{q_now} Target"].where(parts[f"Q{q_now} Target"] > 0)
    S2 = S.assign(Policy=S["Partner"].map(cat).fillna("Old Partner"))
    pol = scorecard(S2, ["Policy"], P, O.assign(Policy=O["Partner"].map(cat).fillna("Old Partner")),
                    B.assign(Policy=B["Partner"].map(cat).fillna("Old Partner"))).reindex(order)
    billed = S2[(S2["Date"] >= P["y_start"]) & (S2["Gross"] > 0)].groupby("Policy")["Partner"].nunique()
    pol.insert(0, "Partners billed YTD", billed.reindex(order).fillna(0))
    pol = with_total(pol)

    # --- QTOD by partner and quarter
    qt = pd.DataFrame()
    if len(pt):
        rows = []
        for q in range(1, 5):
            a = pd.Timestamp(y, 3 * (q - 1) + 1, 1)
            b = min(a + pd.offsets.QuarterEnd(0), P["as_of"])
            qs = S[(S["Date"] >= a) & (S["Date"] <= b)].groupby("Partner")["Value"].sum() if a <= P["as_of"] else pd.Series(dtype=float)
            qtgt = (pt[pt["Quarter"] == f"Q{q}"].set_index("Partner Code")["Target 100%"] * p_rate)
            rows.append(pd.DataFrame({f"Q{q} Target": qtgt, f"Q{q} Sales": qs}))
        qt = pd.concat(rows, axis=1)
        qt = qt.reindex(pt["Partner Code"].unique())
        for q in range(1, 5):
            qt[f"Q{q} Ach %"] = qt[f"Q{q} Sales"] / qt[f"Q{q} Target"].where(qt[f"Q{q} Target"] > 0)
        fam = T["partners"]
        for f_, grp in (("Miluz Zeta", "Miluz Zeta"), ("Wiser", "Wiser")):
            ft = fam[fam["Family"] == f_].groupby("Partner Code")["Target 100%"].sum() * p_rate
            fs = S[(S["Product Group"] == grp) & (S["Date"] >= P["y_start"]) & (S["Date"] <= P["as_of"])].groupby("Partner")["Value"].sum()
            qt[f"{f_} Annual Target"], qt[f"{f_} YTD Sales"] = ft, fs.reindex(qt.index)
            qt[f"{f_} Ach %"] = qt[f"{f_} YTD Sales"] / qt[f"{f_} Annual Target"].where(qt[f"{f_} Annual Target"] > 0)
        info = pt.drop_duplicates("Partner Code").set_index("Partner Code")[["Partner", "Branch", "Region", "Partner type"]]
        qt = info.join(qt)
        qt.index.name = "Partner Code"

    # --- SSP
    ssp_t = T["ssp"]
    ssp = pd.DataFrame()
    if len(ssp_t):
        st = ssp_t[ssp_t["Year"] == y].copy()
        st["Target"] = st["Target"].map(lambda v: to_basis(v, "Retail", basis, y))
        bs = scorecard(S, ["Branch"], P)
        ssp = st.groupby(["SSP", "Branch"])["Target"].sum().to_frame("Yearly Target")
        ssp["YTD Sales"] = [bs["YTD Sales"].get(b, 0.0) for _, b in ssp.index]
        ssp["YTD Sales LY"] = [bs["YTD Sales LY"].get(b, 0.0) for _, b in ssp.index]
        ssp["YTD Growth %"] = ssp["YTD Sales"] / ssp["YTD Sales LY"].where(ssp["YTD Sales LY"] > 0) - 1
        ssp["YTD Ach %"] = ssp["YTD Sales"] / ssp["Yearly Target"].where(ssp["Yearly Target"] > 0)
        ssp = ssp.reset_index().set_index(["SSP", "Branch"])
    ssa = D["masters"]["ssa"]
    with_ssp = pd.DataFrame()
    if ssa:
        S3 = S.assign(**{"SSP coverage": S["Branch"].astype(str).str.upper().map(ssa).fillna("No SSP")})
        S3["SSP coverage"] = np.where(S3["SSP coverage"].astype(str).str.lower().isin(["no ssa", "no ssp", "nan", ""]), "Without SSP", "With SSP")
        with_ssp = with_total(scorecard(S3, ["SSP coverage"], P))

    # --- MTD daily
    d = S[(S["Date"] >= P["m_start"]) & (S["Date"] <= P["as_of"])]
    daily = d.groupby(d["Date"].dt.normalize()).agg(**{"Sales": ("Value", "sum"), "Partners billed": ("Partner", "nunique"),
                                               "Lines": ("Value", "size")})
    daily.index.name = "Date"
    daily["Cumulative Sales"] = daily["Sales"].cumsum()

    # --- orders
    ob_trend = month_trend(O, P)
    ob_trend.columns = [c.replace("Sales", "OB") for c in ob_trend.columns]

    yearly_total = yt.sum() if len(yt) else np.nan
    pg = with_total(scorecard(S, ["Product Group"], P, O, B, qty=True))
    pm = T["prod_m"]
    if len(pm):
        pmt = pm.assign(Target=pm["Target"].map(lambda v: to_basis(v, "Retail", basis, y)))
        pmt["Month"] = pd.to_datetime(pmt["Month"])
        ytd_t = pmt[pmt["Month"] <= P["as_of"]].groupby("Product group")["Target"].sum()
        pg["YTD Product Target (Zeta / Wiser)"] = ytd_t.reindex(pg.index)
        pg["YTD Product Ach %"] = pg["YTD Sales"] / pg["YTD Product Target (Zeta / Wiser)"].where(pg["YTD Product Target (Zeta / Wiser)"] > 0)
    tot = scorecard(S.assign(All="Retail"), ["All"], P, O.assign(All="Retail"), B.assign(All="Retail"))
    r0 = tot.iloc[0] if len(tot) else pd.Series(dtype=float)
    mtd_t = mt.sum() if len(mt) else np.nan
    active = lambda a, b: S[(S["Date"] >= a) & (S["Date"] <= b) & (S["Gross"] > 0)]["Partner"].nunique()
    geo = new_geo(S, P)
    overview = kpis([
        (f"YTD Sales{M_}", r0.get("YTD Sales", 0) / UNIT, f"1 Jan – {P['label']}"),
        (f"YTD Sales last year{M_}", r0.get("YTD Sales LY", 0) / UNIT, "same days last year"),
        ("YTD Growth %", r0.get("YTD Growth %"), ""),
        (f"Yearly Target{M_}", yearly_total / UNIT, "Branch Targets"),
        ("YTD Ach % of yearly target", _ach(r0.get("YTD Sales", 0), yearly_total), f"{(12 - P['months_left']) / 12:.0%} of the year gone"),
        (f"Needed per month{M_}", max(yearly_total - r0.get("YTD Sales", 0), 0) / max(P["months_left"], 1) / UNIT, f"{P['months_left']} month(s) left"),
        (f"{mon} Sales (MTD){M_}", r0.get("MTD Sales", 0) / UNIT, f"{P['m_start']:%d %b} – {P['label']}"),
        (f"{mon} Target{M_}", mtd_t / UNIT, "Monthly Targets"),
        (f"{mon} Ach %", _ach(r0.get("MTD Sales", 0), mtd_t), ""),
        (f"{mon} Growth %", r0.get("MTD Growth %"), "vs the same days last year"),
        (f"YTD OB{M_}", r0.get("YTD OB", 0) / UNIT, ""),
        (f"Open Backlog{M_}", r0.get("Open Backlog", 0) / UNIT, ""),
        ("Active partners YTD", active(P["y_start"], P["as_of"]), f"last year {active(P['ly_y_start'], P['ly_as_of'])}"),
        (f"New partners (policy) — {order[0]}", int((cat == order[0]).sum()), f"new-partner group: {int(bucket.eq('New partner').sum())}"),
        ("Handholding partners", int(bucket.eq("Handholding").sum()), order[3]),
        ("New geo cities", len(geo), "sales this year, none in the two years before"),
    ])
    overview.columns = ["Value", "Note"]

    rg = with_total(scorecard(S, ["Region"], P, O, B, rg_t(mt) if len(mt) else None, rg_t(yt) if len(yt) else None))
    br = scorecard(S, ["Branch"], P, O, B, mt, yt)
    br.insert(0, "Region", br.index.map(lambda b: region_of.get(b, "Unmapped")))
    br.insert(1, "Active partners YTD", S[(S["Date"] >= P["y_start"]) & (S["Gross"] > 0)].groupby("Branch")["Partner"].nunique().reindex(br.index))
    br = with_total(br.sort_values(["Region", "YTD Sales"], ascending=[True, False]))
    bm = []
    for (b, m), tv in mt_all.items():
        if m > P["as_of"]:
            continue
        e = min(m + pd.offsets.MonthEnd(0), P["as_of"])
        sv = _sum(S[S["Branch"] == b], m, e)
        lv = _sum(S[S["Branch"] == b], m.replace(year=y - 1), (m + pd.offsets.MonthEnd(0)).replace(year=y - 1))
        bm.append({"Branch": b, "Region": region_of.get(b, "Unmapped"), "Month": m.strftime("%b %Y"), "Month Target": tv,
                   "Sales": sv, "Ach %": _ach(sv, tv), "Sales LY": lv, "Growth %": _pct(sv, lv)})
    bm = pd.DataFrame(bm).set_index(["Branch", "Region", "Month"]) if bm else pd.DataFrame()

    sheets = [
        ("About", f"Retail analysis — {basis}", about_sheet("Retail", basis, P, T), None),
        ("Overview", f"Retail — overview ({basis}, INR {UNIT_LABEL})", overview, f"As of {P['label']}"),
        ("Monthly trend", "Retail — sales by month vs last years, monthly target and yearly target progress",
         month_trend(S, P, month_tot, yearly_total), "Month Target = Monthly Targets (commit); YTD Ach % = YTD ÷ yearly target"),
        ("Regions", "Retail — regions", rg, "Region of each branch as in Branch Targets"),
        ("Branches", "Retail — branches", br, None),
        ("Branch x month", "Retail — monthly target vs sales by branch", bm, "Monthly Targets as given; months without a target are not shown"),
        ("Products", "Retail — product groups (value and quantity)", pg, "Product targets only where given (Zeta / Wiser monthly)"),
        ("Families", "Retail — families", with_total(scorecard(S, ["Family"], P, O, B, qty=True).sort_values("YTD Sales", ascending=False)), None),
        ("Branch x family", "Retail — YTD by branch and product group", crosstab(S, "Branch", "Product Group", P), None),
        ("Branch x family qty", "Retail — YTD quantity by branch and product group", crosstab(S, "Branch", "Product Group", P, col="Qty"), None),
        ("Activities", "Retail — activities", with_total(scorecard(S, ["Activity"], P, O, B)), None),
        ("RD vs KRO", "Retail — partner type", with_total(scorecard(S, ["Partner type"], P, O, B)), "RD = pricing class ERW, KRO = EKR"),
        ("City tier", "Retail — city tier", with_total(scorecard(S, ["Tier"], P, O, B)), "Tier from the Partner Master"),
        ("Partners", "Retail — all partners", parts, "Policy category and assigned target as the Partner Analysis policy tracker"),
        ("Partner targets", "Retail — QTOD / ATOD by partner and quarter", qt, "Targets from the QTOD and ATOD file"),
        ("New partners", "Retail — partners by policy category", pol, "New Partner = sales this year, none in the two years before"),
        ("New partner list", "Retail — new and onboarded partners",
         parts[parts["Policy group"].isin(["New partner", "Handholding"])] if "Policy group" in parts else pd.DataFrame(), None),
        ("New geo", "Retail — new cities", geo, "Cities with sales this year and none in the two years before"),
        ("SSP", "Retail — SSP target vs achievement", ssp, "Achievement = sales of the SSP's branch"),
        ("With vs without SSP", "Retail — sales in branches with / without an SSP", with_ssp, "SSP Mapping of the Data Dictionary"),
        ("MTD", f"Retail — {mon} by branch", scorecard(S, ["Branch"], P, None, None, mt).sort_values("MTD Sales", ascending=False), None),
        ("MTD daily", f"Retail — {mon} day by day", daily, None),
        ("Order book", "Retail — order booking by month", ob_trend, None),
        ("Backlog", "Retail — open backlog by branch and age", backlog_ageing(B, ["Branch"], P), "Age = days since order creation"),
        ("Backlog by partner", "Retail — open backlog by partner (top 100)",
         backlog_ageing(B, ["Partner", "Partner name"], P).head(100), None),
        ("Target issues", "Problems found in the target files", T["issues"].set_index("Source sheet") if len(T["issues"]) else None, None),
    ]
    return sheets


def hs_book(D, T, P, basis, with_fd):
    y, mon = P["year"], P["mon"]
    keep = (lambda d: d) if with_fd else (lambda d: d[d["Activity"] != "FD"])
    S, O, B = (keep(with_values(D[k], basis, P)) for k in ("hs_sales", "hs_ob", "hs_bl"))
    tag = "with FD" if with_fd else "without FD"
    region_of = T["region_of"]
    # targets: Retail yearly / monthly (with FD only), Project yearly by activity, commits by activity
    r_yt = T["yearly"].map(lambda v: to_basis(v, "Retail", basis, y))
    r_mt_all = T["monthly"].map(lambda v: to_basis(v, "Retail", basis, y))
    py = T["project_yearly"].copy()
    if len(py):
        py["Target"] = py["Target"].map(lambda v: to_basis(v, "Project", basis, y))
    com = T["commits"].copy()
    if len(com):
        com["Commit"] = [to_basis(v, g, basis, y) for v, g in zip(com["Commit"], com["GTM"])]
        com = com[(com["GTM"] == "Retail") | (com["Activity"] != "FD")]
        if not with_fd:
            com = com[com["Activity"] != "FD"]
    m_key = P["m_start"]
    r_mt = r_mt_all.xs(m_key, level="Month") if m_key in r_mt_all.index.get_level_values(1) else pd.Series(dtype=float)
    p_yt = py.groupby("Branch")["Target"].sum() if len(py) else pd.Series(dtype=float)
    p_mt = (com.loc[(com["GTM"] == "Project") & (pd.to_datetime(com["Month"]) == m_key), "Commit"].sum()
            if len(com) else np.nan) or np.nan                     # Project month target = this month's commits

    # Retail / Project / overall
    def seg(sel, yt=np.nan, mt=np.nan):
        s, o, b = S[sel(S)], O[sel(O)], B[sel(B)]
        t = scorecard(s.assign(All=1), ["All"], P, o.assign(All=1), b.assign(All=1)).iloc[0] if len(s) else pd.Series(dtype=float)
        t = t.copy()
        t["Month Target"], t["Yearly Target"] = mt, yt
        t["Month Ach %"], t["YTD Ach %"] = _ach(t.get("MTD Sales", 0), mt), _ach(t.get("YTD Sales", 0), yt)
        return t
    ret, proj, fd = (lambda d: d["GTM"] == "Retail"), (lambda d: d["GTM"] == "Project"), (lambda d: d["Activity"] == "FD")
    r_y, r_m = (r_yt.sum(), r_mt.sum()) if with_fd else (np.nan, np.nan)
    rows = {}
    if with_fd:
        rows["Retail – without FD"] = seg(lambda d: ret(d) & ~fd(d))
        rows["Retail – FD only"] = seg(lambda d: ret(d) & fd(d))
    rows["Retail – total" if with_fd else "Retail (without FD)"] = seg(ret, r_y, r_m)
    rows["HS Project"] = seg(proj, p_yt.sum() if len(p_yt) else np.nan, p_mt)
    rows[f"HS overall (Retail + Project, {tag})"] = seg(lambda d: pd.Series(True, index=d.index),
                                                        (r_y + p_yt.sum()) if with_fd and len(p_yt) else np.nan,
                                                        r_m + p_mt if with_fd else np.nan)
    gt = pd.DataFrame(rows).T
    gt.index.name = "Segment"
    gt = gt[[c for c in ["Month Target", "MTD Sales", "Month Ach %", "MTD Growth %", "Yearly Target", "YTD Sales", "YTD Ach %",
                         "YTD Growth %", "YTD Sales LY", f"FY {y - 1} Sales", "YTD OB", "Open Backlog"] if c in gt]]

    # branches: Retail branches (Retail targets) and Project branches (Project targets)
    br = scorecard(S, ["GTM", "Branch"], P, O, B)
    yt_b = pd.Series({("Retail", b): v for b, v in r_yt.items()} if with_fd else {}, dtype=float)
    yt_b = pd.concat([yt_b, pd.Series({("Project", b): v for b, v in p_yt.items()}, dtype=float)])
    br["Yearly Target"] = yt_b.reindex(br.index)
    br["YTD Ach %"] = br["YTD Sales"] / br["Yearly Target"].where(br["Yearly Target"] > 0)
    br["Needed per month"] = (br["Yearly Target"] - br["YTD Sales"]).clip(lower=0) / max(P["months_left"], 1)
    br = with_total(br.sort_values("YTD Sales", ascending=False))
    rg = with_total(scorecard(S, ["Region"], P, O, B))
    act = scorecard(S, ["Activity", "GTM"], P, O, B)
    if len(py):
        act["Yearly Target"] = py.groupby(["Activity", "GTM"])["Target"].sum().reindex(act.index)
        act["YTD Ach %"] = act["YTD Sales"] / act["Yearly Target"].where(act["Yearly Target"] > 0)
    act = with_total(act)
    # commits vs sales by month and activity
    cv = pd.DataFrame()
    if len(com):
        c = com.assign(Month=pd.to_datetime(com["Month"]))
        cs = c.groupby(["Month", "GTM", "Activity"])["Commit"].sum()
        rows = []
        for (m, g, a), v in cs.items():
            e = min(m + pd.offsets.MonthEnd(0), P["as_of"])
            sv = _sum(S[(S["GTM"] == g) & (S["Activity"] == a)], m, e) if m <= P["as_of"] else np.nan
            rows.append({"Month": m.strftime("%b %Y"), "GTM": g, "Activity": a, "Commit": v, "Sales": sv,
                         "Commit Ach %": _ach(sv, v) if pd.notna(v) else np.nan})
        cv = pd.DataFrame(rows).set_index(["Month", "GTM", "Activity"])
    month_tot = r_mt_all.groupby(level="Month").sum().to_dict() if with_fd else None
    trend = month_trend(S, P, month_tot, (r_y + p_yt.sum()) if with_fd and len(p_yt) else None)
    if with_fd:
        trend = trend.rename(columns={"Month Target": "Month Target (Retail only)", "Month Ach %": "Month Ach % (Retail only)"})
    tr_g = pd.concat({g: month_trend(S[S["GTM"] == g], P)[[f"Sales {y}", f"Sales {y - 1}", "Growth % vs LY"]]
                      for g in ("Retail", "Project")}, axis=1)
    tr_g.columns = [f"{g} {c}" for g, c in tr_g.columns]
    parts = partner_table(S, O, B, P)
    wis = lambda d: d[d["Product Group"] == "Wiser"]
    wiser = with_total(scorecard(wis(S), ["GTM"], P, wis(O), wis(B), qty=True), "Wiser total")
    wiser_br = with_total(scorecard(wis(S), ["GTM", "Branch"], P, wis(O), wis(B), qty=True).sort_values("YTD Sales", ascending=False))
    d = S[(S["Date"] >= P["m_start"]) & (S["Date"] <= P["as_of"])]
    daily = d.pivot_table(index=d["Date"].dt.normalize(), columns="GTM", values="Value", aggfunc="sum", fill_value=0.0)
    daily.columns = [f"{c} Sales" for c in daily.columns]
    daily["Total Sales"] = daily.sum(axis=1)
    daily.index.name = "Date"
    ob_trend = month_trend(O, P)
    ob_trend.columns = [c.replace("Sales", "OB") for c in ob_trend.columns]
    tot = gt.iloc[-1]
    overview = kpis([
        (f"YTD Sales{M_}", tot.get("YTD Sales", 0) / UNIT, f"Retail + Project, {tag}, 1 Jan – {P['label']}"),
        ("YTD Growth %", tot.get("YTD Growth %"), "vs the same days last year"),
        (f"Yearly Target{M_}", tot.get("Yearly Target", np.nan) / UNIT,
         "Retail Branch Targets + Project targets" if with_fd else "no target without FD (Retail targets include FD)"),
        ("YTD Ach %", tot.get("YTD Ach %"), ""),
        (f"Retail YTD Sales{M_}", gt.iloc[-3 if with_fd else 0].get("YTD Sales", 0) / UNIT if len(gt) > 2 else np.nan, ""),
        (f"Project YTD Sales{M_}", gt.loc["HS Project", "YTD Sales"] / UNIT if "HS Project" in gt.index else np.nan, ""),
        ("Project YTD Ach %", gt.loc["HS Project", "YTD Ach %"] if "HS Project" in gt.index else np.nan, "Project yearly targets (HS file)"),
        (f"{mon} Sales (MTD){M_}", tot.get("MTD Sales", 0) / UNIT, ""),
        (f"{mon} Growth %", tot.get("MTD Growth %"), ""),
        (f"YTD OB{M_}", tot.get("YTD OB", 0) / UNIT, ""),
        (f"Open Backlog{M_}", tot.get("Open Backlog", 0) / UNIT, ""),
        ("Retail share of YTD Sales %", _ach(S.loc[(S["GTM"] == "Retail") & (S["Date"] >= P["y_start"]), "Value"].sum(),
                                             S.loc[S["Date"] >= P["y_start"], "Value"].sum()), ""),
        ("Active partners YTD", S[(S["Date"] >= P["y_start"]) & (S["Gross"] > 0)]["Partner"].nunique(), ""),
    ])
    overview.columns = ["Value", "Note"]
    scope = f"HS (Retail + Project) {tag}"
    return [
        ("About", f"{scope} — {basis}", about_sheet(scope, basis, P, T), None),
        ("Overview", f"{scope} — overview ({basis}, INR {UNIT_LABEL})", overview, f"As of {P['label']}"),
        ("Retail vs Project", f"{scope} — Retail and Project", gt,
         "Month Target: Retail = Monthly Targets, Project = this month's commits (HS targets file)"
         + ("" if with_fd else "; Retail targets include FD, so they are not shown without FD")),
        ("Monthly trend", f"{scope} — sales by month vs last years", trend,
         "Month Target = Retail Monthly Targets only (Project has no monthly target)" if with_fd else None),
        ("Monthly trend by GTM", f"{scope} — Retail and Project by month", tr_g, None),
        ("Regions", f"{scope} — regions", rg, "Retail rows: region of the branch; Project rows: sales-office region"),
        ("Branches", f"{scope} — branches (Retail branch / Project sales office)", br, None),
        ("Activities", f"{scope} — activities", act, "Yearly target = Project targets by activity (HS file)"),
        ("Commit vs sales", f"{scope} — monthly commits vs sales by activity", cv,
         "Commits as in the HS targets file (Project FD commits left out); months without a commit are not shown"),
        ("Products", f"{scope} — product groups", with_total(scorecard(S, ["Product Group"], P, O, B, qty=True).sort_values("YTD Sales", ascending=False)), None),
        ("Families", f"{scope} — families", with_total(scorecard(S, ["Family"], P, O, B, qty=True).sort_values("YTD Sales", ascending=False)), None),
        ("Branch x family", f"{scope} — YTD by branch and product group", crosstab(S, "Branch", "Product Group", P), None),
        ("GTM x family", f"{scope} — YTD by Retail / Project and product group", crosstab(S, "Product Group", "GTM", P), None),
        ("Partners", f"{scope} — all partners", parts, None),
        ("Wiser", f"{scope} — Wiser", wiser, None),
        ("Wiser by branch", f"{scope} — Wiser by branch", wiser_br, None),
        ("MTD", f"{scope} — {mon} by branch", scorecard(S, ["GTM", "Branch"], P).sort_values("MTD Sales", ascending=False), None),
        ("MTD daily", f"{scope} — {mon} day by day", daily, None),
        ("Order book", f"{scope} — order booking by month", ob_trend, None),
        ("Backlog", f"{scope} — open backlog by branch and age", backlog_ageing(B, ["GTM", "Branch"], P), "Age = days since order creation"),
        ("Target issues", "Problems found in the target files", T["issues"].set_index("Source sheet") if len(T["issues"]) else None, None),
    ]

In [ ]:
# ════════════════════════════════════════════════════════════════════
# 10 · RUN — reads the data once and writes the six workbooks
# ════════════════════════════════════════════════════════════════════
t0 = time.time()
DATA = load_data()
PERIOD = period(DATA)
TARGETS = targets(DATA, PERIOD)
stamp = f"{PERIOD['mon']}_{PERIOD['year']}"
written = []
for basis in ("Gross", "Net"):
    written.append(write_book(f"Retail_Analysis_{basis}_{stamp}.xlsx", retail_book(DATA, TARGETS, PERIOD, basis), PERIOD))
    for with_fd in (True, False):
        tag = "with_FD" if with_fd else "without_FD"
        written.append(write_book(f"HS_{tag}_Analysis_{basis}_{stamp}.xlsx", hs_book(DATA, TARGETS, PERIOD, basis, with_fd), PERIOD))
print(f"\n✅ {len(written)} workbooks as of {PERIOD['label']} in {time.time() - t0:.0f}s:")
for w in written:
    print("   ", w)